# VoiceLK TTS — Full Evaluation Suite

Evaluates every VITS run stored in `model_evaluate/model/` and writes every artifact to
`model_evaluate/outputs/`.

**Coverage** (metrics requested for the VoiceLK evaluation):

| Group | Metric | Where |
|---|---|---|
| Objective | RTF, latency percentiles | §6 |
| Objective | Mel-Cepstral Distortion (MCD) | §7 |
| Objective | F0 RMSE / pitch correlation / voicing error | §8 |
| Objective | PESQ, STOI / ESTOI | §9 |
| Objective | ASR-based WER / CER (with ground-truth calibration) | §10 |
| Objective | Neural MOS predictor (UTMOS) | §11 |
| Objective | Speaker similarity + speaker consistency | §12 |
| Objective | Prosody / duration alignment | §13 |
| Component | G2P accuracy + IPA→vocab coverage | §14 |
| Component | Language-routing (code-switch) accuracy | §15 |
| Component | Lexicon coverage rate | §16 |
| System | End-to-end latency under load | §17 |
| System | Robustness / edge cases (numbers, acronyms, mixed script) | §18 |
| System | Model size / footprint | §3 |
| Subjective | MOS, CMOS, AB/ABX, transcription/intelligibility, MUSHRA, Turing-test, ICT comprehension | §19 |
| Analysis | Error taxonomy, A/B deployment spec, summary report | §20–§22 |

Metrics that cannot be computed on this machine are **not silently dropped** — each one is
recorded in `91_skipped_metrics.csv` together with the reason and what it would take to run it.

**Kernel:** run this with the `venv_evoluation` interpreter
(`D:\RUSL\Final Project\TTS\voicelk_ml\venv_evoluation\Scripts\python.exe`), which already has
torch, librosa, soundfile, tensorboard, scipy, sklearn and the vendored-TTS runtime deps.
Optional extras (PESQ, STOI, ASR, speaker encoder) are installed from §0.1 — everything that
depends on them degrades to a recorded skip if they are missing.

**Run order:** top to bottom. Later cells depend on objects built earlier (`RUNS`, `MODELS`,
`EVAL_SENTENCES`, `REF_PAIRS`, `RESULTS`).

---
## 0. Environment, configuration and capability probe

In [ ]:
import os, sys, json, glob, time, math, hashlib, random, warnings, platform, unicodedata, textwrap
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="torch")

# ---------------------------------------------------------------- paths
VOICELK_ML = r"D:\RUSL\Final Project\TTS\voicelk_ml"
EVAL_DIR   = os.path.join(VOICELK_ML, "model_evaluate")
MODEL_ROOT = os.path.join(EVAL_DIR, "model")
OUT_DIR    = os.path.join(EVAL_DIR, "outputs")
DATA_DIR   = os.path.join(VOICELK_ML, "data")                 # metadata (.txt/.csv)
os.makedirs(OUT_DIR, exist_ok=True)

# Ground-truth wav roots are searched in order; the first one holding a "wavs" folder wins.
GT_ROOT_CANDIDATES = [
    os.path.dirname(VOICELK_ML),                  # ...\TTS        -> ...\TTS\data\wavs
    VOICELK_ML,
]

# Vendored Coqui TTS + VoiceLK NLP front-end (neither is pip-installed)
sys.path.insert(0, os.path.join(VOICELK_ML, "model_training"))
sys.path.insert(0, os.path.join(VOICELK_ML, "model_engine"))

# ---------------------------------------------------------------- knobs
CONFIG = {
    # Which checkpoint to evaluate per run: "auto" picks best_model.pth, else the
    # highest-step best_model_*.pth, else the highest-step checkpoint_*.pth.
    "primary_checkpoint": "auto",
    "device": "auto",                  # "auto" | "cpu" | "cuda"
    "common_sr": 16000,                # every cross-model / reference comparison resamples here
    "speaker_id": 0,                   # used only by runs with use_speaker_embedding=True

    "footprint_all_checkpoints": True, # torch.load every checkpoint (slow: ~1 GB each)
    "n_ref_pairs": 8,                  # held-out ground-truth pairs for MCD/F0/PESQ/STOI/ASR
    "n_train_ref_pairs": 16,           # extra pairs sampled from the TRAINING split (see §5)
    "rtf_repeats": 3,                  # timed repeats per sentence
    "load_concurrency": [1, 2, 4],     # threads for the latency-under-load test
    "load_requests": 12,               # requests per concurrency level

    "enable_asr": True,
    "asr_model_id": "SpideyDLK/wav2vec2-large-xls-r-300m-sinhala-low-LR-part1",
    "enable_utmos": True,
    "enable_speaker_encoder": True,
    "speaker_encoder_id": "speechbrain/spkrec-ecapa-voxceleb",

    "build_listening_kits": True,      # writes wavs + rating sheets for the human tests
    "random_seed": 1234,
}
random.seed(CONFIG["random_seed"])
np.random.seed(CONFIG["random_seed"])

# ---------------------------------------------------------------- capability probe
def _probe(name, importer):
    try:
        importer()
        return True, ""
    except Exception as exc:
        return False, f"{type(exc).__name__}: {exc}"

CAPS, CAP_ERR = {}, {}
_probes = {
    "torch":       lambda: __import__("torch"),
    "librosa":     lambda: __import__("librosa"),
    "soundfile":   lambda: __import__("soundfile"),
    "scipy":       lambda: __import__("scipy.signal", fromlist=["signal"]),
    "matplotlib":  lambda: __import__("matplotlib"),
    "tensorboard": lambda: __import__("tensorboard"),
    "pesq":        lambda: __import__("pesq"),
    "pystoi":      lambda: __import__("pystoi"),
    "transformers":lambda: __import__("transformers"),
    "speechbrain": lambda: __import__("speechbrain"),
}
for _name, _imp in _probes.items():
    CAPS[_name], CAP_ERR[_name] = _probe(_name, _imp)

import torch                      # hard requirement
import soundfile as sf
import librosa
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams["figure.dpi"] = 110

DEVICE = ("cuda" if torch.cuda.is_available() else "cpu") if CONFIG["device"] == "auto" else CONFIG["device"]

print("python      :", platform.python_version(), "|", platform.platform())
print("torch       :", torch.__version__, "| device:", DEVICE,
      "| cuda:", torch.cuda.is_available())
print("numpy/pandas:", np.__version__, "/", pd.__version__)
print("outputs     ->", OUT_DIR)
print("\noptional capabilities")
for _name in _probes:
    print(f"  {_name:13s} {'available' if CAPS[_name] else 'MISSING  -> ' + CAP_ERR[_name][:70]}")

### 0.1 Optional extras

Everything below is optional. Anything still missing when its section runs is recorded as a
skip with a reason instead of failing the notebook.

| Package | Unlocks | Note |
|---|---|---|
| `pystoi` | STOI / ESTOI (§9) | pure Python, installs cleanly |
| `pesq` | PESQ (§9) | C extension — **no cp314 wheel on PyPI**, needs MSVC Build Tools to compile |
| `transformers` | ASR WER/CER (§10) | also downloads a ~1.2 GB Sinhala wav2vec2 model on first use |
| `speechbrain` | ECAPA speaker embeddings (§12) | downloads ~80 MB on first use |

UTMOS (§11) needs no package — it is fetched through `torch.hub` at run time (internet required).

Uncomment and run the cell below to install them into the **currently running kernel**.
If you would rather keep `venv_evoluation` untouched, create a fresh venv inside
`model_evaluate/` and point the kernel at it.

In [ ]:
# %pip install pystoi transformers speechbrain
# %pip install pesq        # needs MS Visual C++ Build Tools on Python 3.14 (no prebuilt wheel)
print("Optional-install cell — uncomment a line above and re-run, then restart the kernel.")

### 0.2 Result and skip registries

Every section appends to `RESULTS` (a flat metric table) or to `SKIPS` (a metric that could not
be measured, with the reason). §22 turns both into `90_summary_metrics.csv`,
`91_skipped_metrics.csv` and `92_evaluation_report.md`.

In [ ]:
RESULTS, SKIPS = [], []

def add_metric(group, metric, model, value, unit="", n=None, note="", higher_is_better=None):
    """Records one number in the flat summary table."""
    RESULTS.append({
        "group": group, "metric": metric, "model": model,
        "value": value, "unit": unit, "n": n,
        "higher_is_better": higher_is_better, "note": note,
    })

def skip_metric(group, metric, reason, needed=""):
    """Records a metric that cannot be measured here, plus what it would take."""
    SKIPS.append({"group": group, "metric": metric, "reason": reason, "what_would_be_needed": needed})
    print(f"  SKIP  {metric}: {reason}")

def save_table(df, name, index=False):
    """Writes a dataframe to outputs/ as UTF-8-BOM csv (so Excel renders Sinhala correctly)."""
    path = os.path.join(OUT_DIR, name)
    df.to_csv(path, index=index, encoding="utf-8-sig")
    print(f"  saved {name}  ({len(df)} rows)")
    return path

def save_json(payload, name):
    path = os.path.join(OUT_DIR, name)
    with open(path, "w", encoding="utf-8") as fh:
        json.dump(payload, fh, ensure_ascii=False, indent=2)
    print(f"  saved {name}")
    return path

def save_fig(fig, name):
    path = os.path.join(OUT_DIR, name)
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    print(f"  saved {name}")
    return path

EVAL_ENVIRONMENT = {
    "evaluated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "platform": platform.platform(),
    "processor": platform.processor(),
    "cpu_count": os.cpu_count(),
    "python": platform.python_version(),
    "torch": torch.__version__,
    "device": DEVICE,
    "cuda_available": bool(torch.cuda.is_available()),
    "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "torch_threads": torch.get_num_threads(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "librosa": librosa.__version__,
    "optional_packages": {name: bool(ok) for name, ok in CAPS.items()},
    "config": {k: v for k, v in CONFIG.items()},
}
save_json(EVAL_ENVIRONMENT, "00_eval_environment.json")
print("registries ready")

---
## 1. Model registry

Discovers every run folder under `model_evaluate/model/` (a run = a folder holding a
`config.json` plus at least one `.pth`), and records the architecture/audio settings that make
the runs comparable or not.

In [ ]:
def _ckpt_step(name):
    digits = "".join(ch for ch in os.path.splitext(name)[0] if ch.isdigit())
    return int(digits) if digits else -1

def pick_primary_checkpoint(names):
    """best_model.pth -> highest-step best_model_*.pth -> highest-step checkpoint_*.pth."""
    if "best_model.pth" in names:
        return "best_model.pth"
    bests = [n for n in names if n.startswith("best_model")]
    if bests:
        return max(bests, key=_ckpt_step)
    ckpts = [n for n in names if n.startswith("checkpoint")]
    if ckpts:
        return max(ckpts, key=_ckpt_step)
    return None

RUNS = {}
for entry in sorted(os.listdir(MODEL_ROOT)):
    run_dir = os.path.join(MODEL_ROOT, entry)
    cfg_path = os.path.join(run_dir, "config.json")
    if not (os.path.isdir(run_dir) and os.path.isfile(cfg_path)):
        continue
    with open(cfg_path, encoding="utf-8") as fh:
        cfg = json.load(fh)
    ckpts = sorted(n for n in os.listdir(run_dir)
                   if n.endswith(".pth") and "speakers" not in n.lower())
    primary = (pick_primary_checkpoint(ckpts) if CONFIG["primary_checkpoint"] == "auto"
               else CONFIG["primary_checkpoint"])
    ma, audio = cfg.get("model_args", {}), cfg.get("audio", {})
    RUNS[entry] = {
        "run": entry,
        "dir": run_dir,
        "config": cfg,
        "checkpoints": ckpts,
        "primary_checkpoint": primary,
        "run_name": cfg.get("run_name"),
        "formatter": (cfg.get("datasets") or [{}])[0].get("formatter"),
        "meta_file": (cfg.get("datasets") or [{}])[0].get("meta_file_train"),
        "sample_rate": audio.get("sample_rate"),
        "mel_fmax": audio.get("mel_fmax"),
        "num_chars": ma.get("num_chars"),
        "num_speakers": ma.get("num_speakers"),
        "use_speaker_embedding": ma.get("use_speaker_embedding"),
        "batch_size": cfg.get("batch_size"),
        "eval_split_size": cfg.get("eval_split_size"),
        "eval_split_max_size": cfg.get("eval_split_max_size"),
        "n_tfevents": len(glob.glob(os.path.join(run_dir, "events.out.tfevents.*"))),
    }

registry_df = pd.DataFrame([{k: v for k, v in r.items() if k not in ("config", "checkpoints", "dir")}
                            | {"n_checkpoints": len(r["checkpoints"])}
                            for r in RUNS.values()])
save_table(registry_df, "01_model_registry.csv")
print()
for name, r in RUNS.items():
    print(f"{name}\n    primary : {r['primary_checkpoint']}   ({len(r['checkpoints'])} checkpoints, "
          f"{r['n_tfevents']} tfevents)\n    audio   : {r['sample_rate']} Hz, mel_fmax={r['mel_fmax']}, "
          f"num_chars={r['num_chars']}, speakers={r['num_speakers']} (embedding={r['use_speaker_embedding']})")
registry_df

> **Reading the registry.** The two `custom` runs share a dataset, a vocabulary and a sample
> rate, so their numbers are directly comparable. The `pathnirwana` run is a different dataset
> at a different sample rate with a speaker-embedding table, so treat any comparison against it
> as context, not as an A/B of the same system.

---
## 2. Model size / footprint

Per checkpoint: file size, parameter count, training step/epoch reached, and the train/eval loss
stored inside it. Byte-identical checkpoints (a `best_model.pth` that is just a copy of a
`best_model_<step>.pth`) are detected by MD5 so they are not reported as two separate models.

In [ ]:
def md5_of_file(path, block=1 << 24):
    h = hashlib.md5()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(block), b""):
            h.update(chunk)
    return h.hexdigest()

ckpt_rows = []
for run_name, run in RUNS.items():
    names = run["checkpoints"] if CONFIG["footprint_all_checkpoints"] else [run["primary_checkpoint"]]
    for name in names:
        path = os.path.join(run["dir"], name)
        size_mb = os.path.getsize(path) / 1024 ** 2
        row = {"run": run_name, "checkpoint": name, "size_mb": round(size_mb, 1),
               "is_primary": name == run["primary_checkpoint"]}
        try:
            state = torch.load(path, map_location="cpu", weights_only=False)
            weights = state.get("model", {})
            n_params = sum(int(v.numel()) for v in weights.values() if hasattr(v, "numel"))
            row.update({
                "n_params": n_params,
                "params_fp32_mb": round(n_params * 4 / 1024 ** 2, 1),
                "step": state.get("step"), "epoch": state.get("epoch"), "date": state.get("date"),
            })
            loss = state.get("model_loss") or {}
            if isinstance(loss, dict):
                row["train_loss"] = loss.get("train_loss")
                row["eval_loss"] = loss.get("eval_loss")
            elif loss:
                row["train_loss"] = loss
            del state, weights
        except Exception as exc:
            row["load_error"] = f"{type(exc).__name__}: {exc}"
        row["md5"] = md5_of_file(path)
        ckpt_rows.append(row)
        print(f"  {run_name[:34]:34s} {name:24s} {row['size_mb']:7.1f} MB  step={row.get('step')}")

ckpt_df = pd.DataFrame(ckpt_rows)
dupes = ckpt_df[ckpt_df.duplicated("md5", keep=False)].sort_values("md5")
save_table(ckpt_df, "02_checkpoint_footprint.csv")

print("\nbyte-identical checkpoint groups:")
if dupes.empty:
    print("  none")
else:
    for h, grp in dupes.groupby("md5"):
        print("  " + h[:10] + " -> " + ", ".join(f"{r.run}/{r.checkpoint}" for r in grp.itertuples()))

for run_name, run in RUNS.items():
    prim = ckpt_df[(ckpt_df.run == run_name) & ckpt_df.is_primary]
    if prim.empty:
        continue
    p = prim.iloc[0]
    add_metric("system", "model_size_on_disk", run_name, float(p["size_mb"]), "MB",
               note=f"checkpoint={p['checkpoint']} (optimizer state included)", higher_is_better=False)
    if "n_params" in p and pd.notna(p.get("n_params")):
        add_metric("system", "parameters", run_name, int(p["n_params"]), "params", higher_is_better=False)
        add_metric("system", "weights_only_fp32", run_name, float(p["params_fp32_mb"]), "MB",
                   note="weights only — deployable footprint without optimizer state",
                   higher_is_better=False)

fig, ax = plt.subplots(figsize=(9, 0.4 * len(ckpt_df) + 1.5))
labels = [f"{r.run[:22]}/{r.checkpoint}" for r in ckpt_df.itertuples()]
ax.barh(labels, ckpt_df["size_mb"], color=["#2a6f97" if p else "#a8c7dc" for p in ckpt_df["is_primary"]])
ax.set_xlabel("checkpoint size (MB)")
ax.set_title("Checkpoint footprint — dark = primary checkpoint evaluated below")
ax.invert_yaxis()
ax.tick_params(labelsize=7)
save_fig(fig, "02_checkpoint_footprint.png")
plt.show()
ckpt_df.head(20)

> A `.pth` here is ~950 MB because Coqui stores the optimizer state next to the weights.
> `weights_only_fp32` is the number that matters for deployment — export with
> `torch.save(model.state_dict(), ...)` (optionally fp16) before shipping.

### 2.1 Speaker-embedding table health

A run with `use_speaker_embedding=True` carries an `emb_g.weight` table of
`num_speakers × 256`. If `num_speakers` is larger than the number of speakers that actually
appear in the metadata, most of those rows never receive a gradient and keep their random
initialisation — and synthesising with an untrained row produces an arbitrary voice.

This cell checks which rows moved away from initialisation, by row norm against the bulk
distribution (robust z-score on the median/MAD). It matters for interpreting the results,
because the notebook synthesises with `CONFIG["speaker_id"]` and that index has to be a
**trained** row for the numbers to describe the intended voice.

In [ ]:
emb_rows = []
for run_name, run in RUNS.items():
    ma = run["config"].get("model_args", {})
    if not ma.get("use_speaker_embedding"):
        print(f"  {run_name[:40]:40s} no speaker embedding table (single-speaker architecture)")
        continue
    path = os.path.join(run["dir"], run["primary_checkpoint"])
    state = torch.load(path, map_location="cpu", weights_only=False)
    weight = state["model"].get("emb_g.weight")
    del state
    if weight is None:
        print(f"  {run_name[:40]:40s} emb_g.weight absent from the checkpoint")
        continue
    weight = weight.detach().float().numpy()
    norms = np.linalg.norm(weight, axis=1)
    median = float(np.median(norms))
    mad = float(np.median(np.abs(norms - median))) or 1e-9
    z = (norms - median) / mad
    trained = np.where(np.abs(z) > 5)[0]                 # moved clearly off the bulk
    selected = CONFIG["speaker_id"]
    emb_rows.append({
        "run": run_name,
        "declared_num_speakers": ma.get("num_speakers"),
        "embedding_dim": weight.shape[1],
        "rows_off_initialisation": len(trained),
        "trained_row_indices": ",".join(str(i) for i in trained[:12]),
        "bulk_norm_median": round(median, 3),
        "selected_speaker_id": selected,
        "selected_row_norm": round(float(norms[selected]), 3),
        "selected_row_z": round(float(z[selected]), 1),
        "selected_row_is_trained": bool(abs(z[selected]) > 5),
        "unused_rows": int(weight.shape[0] - max(len(trained), 1)),
        "wasted_params": int((weight.shape[0] - max(len(trained), 1)) * weight.shape[1]),
    })
    row = emb_rows[-1]
    print(f"  {run_name[:40]:40s} table {weight.shape[0]}x{weight.shape[1]}, "
          f"{len(trained)} row(s) off initialisation, speaker_id={selected} "
          f"z={row['selected_row_z']:+.1f} "
          f"({'TRAINED' if row['selected_row_is_trained'] else 'UNTRAINED - suspect'})")

if emb_rows:
    emb_df = pd.DataFrame(emb_rows)
    save_table(emb_df, "02_speaker_embedding_health.csv")
    for row in emb_rows:
        add_metric("component", "speaker_embedding_rows_trained", row["run"],
                   row["rows_off_initialisation"], "rows",
                   n=row["declared_num_speakers"],
                   note=f"of {row['declared_num_speakers']} declared; "
                        f"speaker_id={row['selected_speaker_id']} is "
                        f"{'trained' if row['selected_row_is_trained'] else 'UNTRAINED'}",
                   higher_is_better=None)
        if not row["selected_row_is_trained"]:
            skip_metric("objective", f"valid speaker selection[{row['run']}]",
                        f"CONFIG['speaker_id']={row['selected_speaker_id']} points at an embedding row "
                        "that never left its random initialisation, so this run's audio is not the "
                        "trained voice",
                        "find the trained row index in 02_speaker_embedding_health.csv and set "
                        "CONFIG['speaker_id'] to it, then re-run")

---
## 3. Training curves (TensorBoard scalars)

Each run's `events.out.tfevents.*` files are parsed, concatenated and de-duplicated by step
(Kaggle/Colab session reconnects produce overlapping event files). Runs with no event files are
recorded as a skip.

In [ ]:
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

SCALARS = {}
for run_name, run in RUNS.items():
    files = sorted(glob.glob(os.path.join(run["dir"], "events.out.tfevents.*")))
    if not files:
        skip_metric("training", f"training_curves[{run_name}]",
                    "no TensorBoard event files were copied into this run folder",
                    "copy events.out.tfevents.* from the training output folder next to the checkpoints")
        continue
    frames = []
    for path in files:
        acc = EventAccumulator(path, size_guidance={"scalars": 0})
        acc.Reload()
        for tag in acc.Tags().get("scalars", []):
            events = acc.Scalars(tag)
            frames.append(pd.DataFrame({
                "wall_time": [e.wall_time for e in events],
                "step": [e.step for e in events],
                "value": [e.value for e in events],
                "tag": tag,
                "source_file": os.path.basename(path),
            }))
    if not frames:
        skip_metric("training", f"training_curves[{run_name}]", "event files contain no scalars", "")
        continue
    df = (pd.concat(frames, ignore_index=True)
            .sort_values(["tag", "step", "wall_time"])
            .drop_duplicates(subset=["tag", "step"], keep="last")
            .reset_index(drop=True))
    SCALARS[run_name] = df
    save_table(df, f"03_training_scalars_{run_name[:28]}.csv")
    hours = (df["wall_time"].max() - df["wall_time"].min()) / 3600
    steps = int(df["step"].max() - df["step"].min())
    print(f"  {run_name}: {len(df)} points, {df['tag'].nunique()} tags, steps {int(df['step'].min())}"
          f"-{int(df['step'].max())}, {hours:.1f} logged hours")
    add_metric("training", "logged_wall_clock", run_name, round(float(hours), 2), "hours",
               note="lower bound — gaps while logging was down are invisible", higher_is_better=False)
    add_metric("training", "throughput", run_name, round(steps / max(hours * 3600, 1), 3), "steps/s",
               higher_is_better=True)

In [ ]:
PLOT_TAGS = ["TrainEpochStats/avg_loss_mel", "EvalStats/avg_loss_mel",
             "TrainEpochStats/avg_loss_kl", "EvalStats/avg_loss_kl",
             "TrainEpochStats/avg_loss_duration", "EvalStats/avg_loss_duration",
             "TrainEpochStats/avg_loss_1", "EvalStats/avg_loss_1"]

for run_name, df in SCALARS.items():
    fig, ax = plt.subplots(figsize=(11, 5.5))
    for tag in PLOT_TAGS:
        sub = df[df["tag"] == tag].sort_values("step")
        if sub.empty:
            continue
        ax.plot(sub["step"], sub["value"], "--" if tag.startswith("EvalStats") else "-",
                label=tag, linewidth=1.2, alpha=0.85)
    ax.set_xlabel("global step"); ax.set_ylabel("loss")
    ax.set_title(f"{run_name} — training / eval losses")
    ax.legend(fontsize=7, ncol=2)
    save_fig(fig, f"03_loss_curves_{run_name[:28]}.png")
    plt.show()

    # train vs eval generalisation gap on the mel loss
    tr = df[df["tag"] == "TrainEpochStats/avg_loss_mel"].sort_values("step")
    ev = df[df["tag"] == "EvalStats/avg_loss_mel"].sort_values("step")
    if not tr.empty and not ev.empty:
        gap = float(ev.tail(5)["value"].mean() - tr.tail(5)["value"].mean())
        last_q = ev[ev["step"] >= ev["step"].max() * 0.75]
        slope = float(np.polyfit(last_q["step"], last_q["value"], 1)[0] * 1000) if len(last_q) > 1 else float("nan")
        best = ev.loc[ev["value"].idxmin()]
        print(f"{run_name}: eval-mel best {best['value']:.3f} @ step {int(best['step'])} | "
              f"train/eval gap {gap:+.3f} | last-quartile slope {slope:+.4f}/1k steps")
        add_metric("training", "train_eval_mel_gap", run_name, round(gap, 3), "loss",
                   note="eval minus train, last 5 logged points; >1.0 suggests overfitting",
                   higher_is_better=False)
        add_metric("training", "best_eval_mel_loss", run_name, round(float(best["value"]), 3), "loss",
                   n=int(best["step"]), higher_is_better=False)

---
## 4. Synthesis engine

One wrapper per run: config → `AudioProcessor` → tokenizer → `Vits` → checkpoint, plus the
VoiceLK front-end (`SinhalaTextNormalizer` → `CodeSwitchedG2P`) so raw Sinhala/English text is
converted exactly the way it was at training time.

`synth()` also reports the **characters the tokenizer silently dropped**: the front-end can emit
IPA symbols (`ə`, `ˈ`, …) that were never in the training vocabulary, and Coqui discards them
without raising. That drop rate is a real pronunciation-failure source and is measured in §14.

In [ ]:
from TTS.tts.configs.vits_config import VitsConfig
from TTS.tts.models.vits import Vits
from TTS.tts.utils.text.tokenizer import TTSTokenizer
from TTS.utils.audio import AudioProcessor
from pipeline import TextProcessingPipeline

TEXT_PIPELINE = TextProcessingPipeline()

class VoiceLKModel:
    """A loaded VITS checkpoint plus everything needed to go from raw text to a waveform."""

    def __init__(self, run, checkpoint=None, device="cpu"):
        self.run_name = run["run"]
        self.run_dir = run["dir"]
        self.checkpoint = checkpoint or run["primary_checkpoint"]
        self.device = device
        cfg_path = os.path.join(self.run_dir, "config.json")
        self.config = VitsConfig()
        self.config.load_json(cfg_path)
        self.ap = AudioProcessor.init_from_config(self.config)
        self.tokenizer, self.config = TTSTokenizer.init_from_config(self.config)
        self.model = Vits(self.config, self.ap, self.tokenizer, speaker_manager=None)
        self.model.load_checkpoint(self.config, os.path.join(self.run_dir, self.checkpoint), eval=True)
        self.model.eval().to(device)
        self.sr = self.config.audio.sample_rate
        self.use_speaker_embedding = bool(self.config.model_args.use_speaker_embedding)
        try:
            self.vocab = set(self.tokenizer.characters.vocab)
        except Exception:
            self.vocab = set()

    def _aux(self):
        speaker_ids = (torch.LongTensor([CONFIG["speaker_id"]]).to(self.device)
                       if self.use_speaker_embedding else None)
        return {"x_lengths": None, "d_vectors": None, "language_ids": None,
                "durations": None, "speaker_ids": speaker_ids}

    def synth_ipa(self, ipa):
        """Synthesises an already-phonemised string (dataset metadata is stored as IPA)."""
        dropped = sorted({c for c in ipa if self.vocab and c not in self.vocab})
        ids = self.tokenizer.text_to_ids(ipa)
        if not ids:
            return {"wav": np.zeros(0, np.float32), "sr": self.sr, "wall_s": 0.0, "audio_s": 0.0,
                    "rtf": float("nan"), "n_tokens": 0, "durations": np.zeros(0),
                    "dropped_chars": dropped, "ipa": ipa, "ok": False, "error": "empty token sequence"}
        x = torch.LongTensor(ids).unsqueeze(0).to(self.device)
        t0 = time.perf_counter()
        with torch.no_grad():
            out = self.model.inference(x, aux_input=self._aux())
        wall = time.perf_counter() - t0
        wav = out["model_outputs"][0, 0].detach().cpu().numpy().astype(np.float32)
        durations = out["durations"][0].detach().cpu().numpy().flatten() if out.get("durations") is not None else np.zeros(0)
        audio_s = len(wav) / self.sr
        return {"wav": wav, "sr": self.sr, "wall_s": wall, "audio_s": audio_s,
                "rtf": wall / audio_s if audio_s > 0 else float("nan"),
                "n_tokens": len(ids), "durations": durations, "dropped_chars": dropped,
                "ipa": ipa, "ok": True, "error": ""}

    def synth(self, text):
        """Full path: raw Sinhala/English text -> normalise -> G2P -> waveform."""
        t0 = time.perf_counter()
        processed = TEXT_PIPELINE.process(text)
        frontend_s = time.perf_counter() - t0
        result = self.synth_ipa(processed["ipa_sequence"])
        result.update({"text": text, "normalized": processed["normalized_text"],
                       "frontend_s": frontend_s, "total_s": frontend_s + result["wall_s"]})
        return result

    def __repr__(self):
        return f"<VoiceLKModel {self.run_name}/{self.checkpoint} @ {self.sr} Hz on {self.device}>"

MODELS = {}
for run_name, run in RUNS.items():
    if not run["primary_checkpoint"]:
        skip_metric("system", f"synthesis[{run_name}]", "run folder has no .pth checkpoint", "")
        continue
    try:
        MODELS[run_name] = VoiceLKModel(run, device=DEVICE)
        print("loaded", MODELS[run_name])
    except Exception as exc:
        skip_metric("system", f"synthesis[{run_name}]", f"checkpoint failed to load: {type(exc).__name__}: {exc}",
                    "check that config.json and the .pth come from the same training run")
print("\nmodels ready:", list(MODELS))

---
## 5. Evaluation corpora

Two corpora are used throughout:

1. **`EVAL_SENTENCES`** — hand-written prompts covering the failure modes VoiceLK cares about
   (plain Sinhala, ICT domain vocabulary, code-switching, acronyms, numbers, dates, URLs,
   symbols, mixed script). Every objective metric that does not need a recording uses this set.
2. **`REF_PAIRS`** — the run's *actual* held-out utterances with their ground-truth recordings.
   Coqui's `split_dataset()` shuffles with `np.random.seed(0)` before slicing, so the eval split
   is reproduced here exactly rather than approximated, which means MCD/F0/PESQ/STOI/ASR are
   measured on audio the model never trained on.
3. **`TRAIN_PAIRS`** — a sample from the *training* split, carried alongside as a second,
   explicitly-labelled condition.

Why (3) exists: `eval_split_size=0.01` over 860 utterances leaves only **8** held-out
recordings, which is too thin to quote a mean and a standard deviation from on its own. Scoring a
larger training-split sample next to it does two useful things — it gives the reference-based
metrics enough samples to be stable, and the difference between the two is a direct
generalisation measure (a model that scores far better on seen sentences has memorised them).

Every table and metric that mixes the two carries a `split` column, and the held-out figure is
always the one reported as the headline result. Training-split numbers are never presented as
model quality on unseen input.

In [ ]:
EVAL_SENTENCES = [
    {"id": "gen01", "category": "general",     "text": "කුඹුර ගොවියාට වී ලබා ගැනීමට උපකාරී වීම් වශයෙන් පිහිට වන්නකි."},
    {"id": "gen02", "category": "general",     "text": "අද කාලගුණය ඉතා අලංකාරයි."},
    {"id": "gen03", "category": "general",     "text": "මගේ නම කුමක්ද කියා ඔබට කිව නොහැක."},
    {"id": "ict01", "category": "ict_domain",  "text": "පරිගණකයක ප්‍රධාන කොටස් තුනකි: ආදාන ඒකක, ක්‍රියාවලි ඒකකය සහ ප්‍රතිදාන ඒකක."},
    {"id": "ict02", "category": "ict_domain",  "text": "දෘඪ තැටිය ස්ථිර ගබඩා ඒකකයක් ලෙස භාවිතා වේ."},
    {"id": "cs01",  "category": "code_switch", "text": "RAM එකක් නැතුව computer එකක් වැඩක් නෑ."},
    {"id": "cs02",  "category": "code_switch", "text": "මේක දන්නැතුව ලංකාවේ OnePlus device එකක් ගන්නවා නම් ඔයා ලොකු වැරැද්දක් කරනවා."},
    {"id": "cs03",  "category": "code_switch", "text": "This is a code-switched sentence with English words."},
    {"id": "acr01", "category": "acronym",     "text": "COVID-19 pandemic එකෙන් පස්සේ WFH වැඩි වුණා."},
    {"id": "acr02", "category": "acronym",     "text": "ICT විෂයේදී CPU සහ GPU අතර වෙනස ඉගෙන ගනිමු."},
    {"id": "num01", "category": "number",      "text": "එක් , දෙක , තුන් , හතර , පහ."},
    {"id": "num02", "category": "number",      "text": "2026 වසරේ සැප්තැම්බර් 08 වන දින පැවැත්වේ."},
    {"id": "num03", "category": "number",      "text": "මිල රුපියල් 1250 ක් වන අතර වට්ටම 15% කි."},
    {"id": "sym01", "category": "symbol",      "text": "a + b = c නම් a < c වේ."},
    {"id": "url01", "category": "url",         "text": "වැඩි විස්තර www.voicelk.lk වෙබ් අඩවියෙන් ලබා ගන්න."},
    {"id": "mix01", "category": "mixed_script","text": "Sinhala සහ English යන භාෂා දෙකම එකම වාක්‍යයක mix කරලා කියනවා."},
    {"id": "long01","category": "long",        "text": "තොරතුරු හා සන්නිවේදන තාක්ෂණය යනු දත්ත එකතු කිරීම, ගබඩා කිරීම, සැකසීම සහ බෙදාහැරීම සඳහා පරිගණක හා සන්නිවේදන උපකරණ භාවිතා කිරීමයි."},
]
eval_df = pd.DataFrame(EVAL_SENTENCES)
save_table(eval_df, "04_eval_corpus.csv")
print(eval_df["category"].value_counts().to_string())
eval_df

In [ ]:
def load_metadata_items(meta_path, wav_root, formatter):
    """Parses a VoiceLK metadata.txt exactly like model_training/formatters.py does.

    custom_formatter uses split("|", 1); the multi-speaker formatters split from the
    outside in (partition/rpartition) so a "|" inside the IPA cannot shift the columns.
    """
    items = []
    multi_speaker = formatter in ("pathnirwana_formatter", "openslr_formatter")
    with open(meta_path, encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            if multi_speaker:
                rel, _, rest = line.partition("|")
                ipa, sep, speaker = rest.rpartition("|")
                if not sep:
                    continue
            else:
                cols = line.split("|", 1)
                if len(cols) < 2:
                    continue
                rel, ipa, speaker = cols[0], cols[1], "voicelk_custom"
            items.append({"audio_file": os.path.join(wav_root, rel.replace("/", os.sep)),
                          "rel_path": rel, "text": ipa, "speaker_name": speaker})
    return items

def reproduce_eval_split(items, eval_split_size=0.01, eval_split_max_size=None):
    """Mirrors TTS.tts.datasets.split_dataset(), including its np.random.seed(0).

    Returns (eval_items, train_items) so the training split can be sampled as a second,
    explicitly-labelled condition (see the note above this cell).
    """
    items = list(items)
    if eval_split_size > 1:
        n_eval = int(eval_split_size)
    elif eval_split_max_size:
        n_eval = min(int(eval_split_max_size), int(len(items) * eval_split_size))
    else:
        n_eval = int(len(items) * eval_split_size)
    n_eval = max(n_eval, 1)

    outer_state = np.random.get_state()
    np.random.seed(0)
    np.random.shuffle(items)
    if len(set(it["speaker_name"] for it in items)) > 1:      # multi-speaker branch
        eval_items, counter = [], Counter(it["speaker_name"] for it in items)
        while len(eval_items) < n_eval and items:
            idx = np.random.randint(0, len(items))
            speaker = items[idx]["speaker_name"]
            if counter[speaker] > 1:
                eval_items.append(items[idx])
                counter[speaker] -= 1
                del items[idx]
        np.random.set_state(outer_state)
        return eval_items, items          # items has had the eval picks removed in-place
    np.random.set_state(outer_state)
    return items[:n_eval], items[n_eval:]

def find_gt_root():
    for base in GT_ROOT_CANDIDATES:
        cand = os.path.join(base, "data")
        if os.path.isdir(os.path.join(cand, "wavs")):
            return cand
    return None

GT_ROOT = find_gt_root()
print("ground-truth audio root:", GT_ROOT)

# raw Sinhala transcripts (used for ASR WER, intelligibility and the listening kits)
ORTHO = {}
ortho_csv = os.path.join(DATA_DIR, "custom_dataset.csv")
if os.path.isfile(ortho_csv):
    _o = pd.read_csv(ortho_csv)
    for row in _o.itertuples():
        ORTHO[str(row.file_name)] = {
            "sinhala": str(getattr(row, "sinhala_transcript", "") or ""),
            "code_switch": str(getattr(row, "sinhala_transcript_with_code_switch", "") or ""),
        }
    print(f"orthographic transcripts: {len(ORTHO)} rows from {os.path.basename(ortho_csv)}")
else:
    print("orthographic transcripts: NOT FOUND ->", ortho_csv)

REF_PAIRS = {}          # run_name -> held-out {audio_file, text(ipa), sinhala, split="heldout"}
TRAIN_PAIRS = {}        # run_name -> training-split sample, split="train"
for run_name, run in RUNS.items():
    meta_name = os.path.basename(run["meta_file"] or "")
    meta_path = os.path.join(DATA_DIR, meta_name)
    if not meta_name or not os.path.isfile(meta_path):
        skip_metric("objective", f"reference_metrics[{run_name}]",
                    f"training metadata '{meta_name}' is not in voicelk_ml/data/",
                    "copy the metadata file this run was trained on into voicelk_ml/data/")
        continue
    if GT_ROOT is None:
        skip_metric("objective", f"reference_metrics[{run_name}]", "no ground-truth wavs/ folder found",
                    "place the dataset's wavs/ folder under TTS/data/ or voicelk_ml/data/")
        continue
def attach_text(items, split):
    out = []
    for it in items:
        if not os.path.isfile(it["audio_file"]):
            continue
        key = os.path.basename(it["rel_path"])
        it = dict(it, split=split)
        it["sinhala"] = ORTHO.get(key, {}).get("sinhala", "")
        it["code_switch"] = ORTHO.get(key, {}).get("code_switch", "")
        out.append(it)
    return out

for run_name, run in RUNS.items():
    meta_name = os.path.basename(run["meta_file"] or "")
    meta_path = os.path.join(DATA_DIR, meta_name)
    if not meta_name or not os.path.isfile(meta_path):
        skip_metric("objective", f"reference_metrics[{run_name}]",
                    f"training metadata '{meta_name}' is not in voicelk_ml/data/",
                    "copy the metadata file this run was trained on into voicelk_ml/data/")
        continue
    if GT_ROOT is None:
        skip_metric("objective", f"reference_metrics[{run_name}]", "no ground-truth wavs/ folder found",
                    "place the dataset's wavs/ folder under TTS/data/ or voicelk_ml/data/")
        continue
    items = load_metadata_items(meta_path, GT_ROOT, run["formatter"])
    held, train = reproduce_eval_split(items, run["eval_split_size"] or 0.01,
                                       run["eval_split_max_size"])
    present = attach_text(held, "heldout")
    if not present:
        skip_metric("objective", f"reference_metrics[{run_name}]",
                    f"none of the {len(held)} held-out recordings for this run exist locally "
                    f"(metadata points at {meta_name}, wavs expected under {GT_ROOT})",
                    "copy this dataset's wavs/ folder to this machine")
        continue
    REF_PAIRS[run_name] = present[: CONFIG["n_ref_pairs"]]
    TRAIN_PAIRS[run_name] = attach_text(train, "train")[: CONFIG["n_train_ref_pairs"]]
    print(f"  {run_name}: {len(present)}/{len(held)} held-out recordings available "
          f"-> using {len(REF_PAIRS[run_name])} held-out + {len(TRAIN_PAIRS[run_name])} training-split")

def ref_sets(run_name):
    """Yields (split, items) for the reference-based metrics: held-out first, then training."""
    for label, store in (("heldout", REF_PAIRS), ("train", TRAIN_PAIRS)):
        items = store.get(run_name) or []
        if items:
            yield label, items

ref_rows = [{"run": r, "split": it["split"], "rel_path": it["rel_path"],
             "has_sinhala_text": bool(it["sinhala"]), "ipa_chars": len(it["text"])}
            for r in RUNS for _, lst in ref_sets(r) for it in lst]
if ref_rows:
    save_table(pd.DataFrame(ref_rows), "04_reference_pairs.csv")

---
## 6. Objective — Real-Time Factor and single-request latency

RTF = wall-clock synthesis time ÷ duration of the audio produced. RTF < 1 means faster than
real time. The front-end (normaliser + G2P) is timed separately from the acoustic model, because
on short prompts the front-end is a meaningful share of the response time.

The first call per model is a warm-up and is excluded — it pays for lazy CUDA/JIT/cache init.

In [ ]:
rtf_rows = []
for run_name, model in MODELS.items():
    model.synth(EVAL_SENTENCES[0]["text"])                     # warm-up, not timed
    for item in EVAL_SENTENCES:
        for rep in range(CONFIG["rtf_repeats"]):
            out = model.synth(item["text"])
            if not out["ok"]:
                continue
            rtf_rows.append({
                "run": run_name, "id": item["id"], "category": item["category"], "repeat": rep,
                "chars": len(item["text"]), "tokens": out["n_tokens"],
                "frontend_ms": out["frontend_s"] * 1000, "model_ms": out["wall_s"] * 1000,
                "total_ms": out["total_s"] * 1000, "audio_s": out["audio_s"],
                "rtf_model": out["rtf"], "rtf_total": out["total_s"] / out["audio_s"],
            })

rtf_df = pd.DataFrame(rtf_rows)
save_table(rtf_df, "10_rtf.csv")

summary = (rtf_df.groupby("run")
           .agg(mean_rtf_model=("rtf_model", "mean"), mean_rtf_total=("rtf_total", "mean"),
                p50_total_ms=("total_ms", lambda s: float(np.percentile(s, 50))),
                p95_total_ms=("total_ms", lambda s: float(np.percentile(s, 95))),
                mean_frontend_ms=("frontend_ms", "mean"), n=("rtf_model", "size"))
           .round(3))
print(summary.to_string(), "\n")
for run_name, row in summary.iterrows():
    add_metric("objective", "rtf_model_only", run_name, round(float(row["mean_rtf_model"]), 3), "ratio",
               n=int(row["n"]), note="acoustic model only; <1 is faster than real time", higher_is_better=False)
    add_metric("objective", "rtf_end_to_end", run_name, round(float(row["mean_rtf_total"]), 3), "ratio",
               n=int(row["n"]), note="front-end + model", higher_is_better=False)
    add_metric("system", "latency_p95_single_request", run_name, round(float(row["p95_total_ms"]), 1), "ms",
               n=int(row["n"]), higher_is_better=False)
    add_metric("system", "frontend_latency_mean", run_name, round(float(row["mean_frontend_ms"]), 1), "ms",
               n=int(row["n"]), note="normaliser + G2P", higher_is_better=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
runs_order = list(rtf_df["run"].unique())
axes[0].boxplot([rtf_df[rtf_df.run == r]["rtf_total"] for r in runs_order],
                tick_labels=[r[:20] for r in runs_order])
axes[0].axhline(1.0, color="crimson", linestyle="--", linewidth=1, label="real time")
axes[0].set_ylabel("RTF (front-end + model)"); axes[0].set_title(f"RTF by run — {DEVICE}")
axes[0].legend(fontsize=8); axes[0].tick_params(axis="x", labelsize=7, rotation=15)
for r in runs_order:
    sub = rtf_df[rtf_df.run == r]
    axes[1].scatter(sub["tokens"], sub["total_ms"], s=14, alpha=0.7, label=r[:20])
axes[1].set_xlabel("input tokens"); axes[1].set_ylabel("total latency (ms)")
axes[1].set_title("Latency vs input length"); axes[1].legend(fontsize=7)
save_fig(fig, "10_rtf.png")
plt.show()
summary

---
## 7. Objective — Mel-Cepstral Distortion (MCD)

MCD is computed against the run's **held-out** recordings and, separately, against a
**training-split** sample (§5), with the synthesised utterance produced from the same IPA string
the recording is labelled with. The held-out mean is the result; the training-split mean is there
for sample size and for the generalisation gap between the two.

The definition used here, stated explicitly so the numbers are reproducible:

* 80-band mel spectrogram (power), `n_fft=1024`, `hop=256`, both signals RMS-normalised to −27 dBFS
* `power_to_db(ref=max, top_db=40)` → natural log, so spectral valleys and the noise floor cannot
  dominate the distance
* DCT-II, scaled by `2/N`; coefficients **c1…c13** (c0 = energy is dropped)
* frames below −40 dB of the utterance peak are excluded (silence vs silence is not informative)
* DTW alignment on the cepstra, then `MCD = (10/ln10) · mean_t sqrt(2 · Σ_i (c_i^ref − c_i^syn)²)`

Sanity anchors measured on this data: identical signals → 0.00 dB, a 30 dB-SNR noisy copy →
0.46 dB, two *different* utterances by the same speaker → ≈16 dB. Because this uses a filterbank
cepstrum rather than an SPTK `mcep` envelope, values are comparable **between the runs in this
notebook**, but should not be quoted directly against published MCD numbers.

In [ ]:
from scipy.fftpack import dct
from scipy.signal import butter, filtfilt

COMMON_SR = CONFIG["common_sr"]
LN10_OVER_10 = np.log(10) / 10.0

def to_sr(wav, sr, target=COMMON_SR):
    wav = np.asarray(wav, dtype=np.float32)
    if wav.ndim > 1:
        wav = wav.mean(axis=1)
    return wav if sr == target else librosa.resample(wav, orig_sr=sr, target_sr=target)

def rms_norm(wav, target_db=-27.0):
    wav = np.asarray(wav, dtype=np.float32)
    return wav * (10 ** (target_db / 20.0) / (float(np.sqrt(np.mean(wav ** 2))) + 1e-12))

def mel_cepstrum(wav, sr=COMMON_SR, n_mels=80, n_coef=13, top_db=40.0, gate_db=-40.0):
    """Natural-log mel cepstrum c1..c_n_coef plus a per-frame 'this frame carries speech' mask."""
    spec = librosa.feature.melspectrogram(y=rms_norm(wav), sr=sr, n_fft=1024, hop_length=256,
                                          n_mels=n_mels, power=2.0)
    log_spec = librosa.power_to_db(spec, ref=np.max, top_db=top_db) * LN10_OVER_10
    coefs = dct(log_spec, axis=0, type=2, norm=None) * (2.0 / n_mels)
    frame_db = librosa.power_to_db(spec.sum(axis=0), ref=np.max)
    return coefs[1:n_coef + 1].T, frame_db > gate_db

def mcd_dtw(ref, syn, sr=COMMON_SR):
    a, mask_a = mel_cepstrum(ref, sr)
    b, mask_b = mel_cepstrum(syn, sr)
    if len(a) < 5 or len(b) < 5:
        return float("nan"), 0
    _, path = librosa.sequence.dtw(X=a.T, Y=b.T, metric="euclidean")
    ia, ib = path[:, 0], path[:, 1]
    keep = mask_a[ia] & mask_b[ib]
    if keep.sum() < 5:
        return float("nan"), int(keep.sum())
    diff = a[ia][keep] - b[ib][keep]
    mcd = (10.0 / np.log(10)) * float(np.mean(np.sqrt(2.0 * np.sum(diff ** 2, axis=1))))
    return mcd, int(keep.sum())

# sanity anchors on the first available reference recording
_anchor_run = next(iter(REF_PAIRS), None)
if _anchor_run:
    _w, _sr = sf.read(REF_PAIRS[_anchor_run][0]["audio_file"])
    _w = to_sr(_w, _sr)
    _noisy = _w + 0.01 * np.random.RandomState(0).randn(len(_w)).astype(np.float32)
    print(f"sanity: self={mcd_dtw(_w, _w)[0]:.2f} dB | +noise={mcd_dtw(_w, _noisy)[0]:.2f} dB")

In [ ]:
SYNTH_CACHE = {}          # (run, rel_path, source) -> synthesis result, reused across §7-§13
PROMPT_CACHE = {}         # (run, prompt id) -> synthesis result for the EVAL_SENTENCES set

def synth_prompt(run_name, item):
    """Cached synthesis of an EVAL_SENTENCES prompt. Never used where timing is measured."""
    key = (run_name, item["id"])
    if key not in PROMPT_CACHE:
        PROMPT_CACHE[key] = MODELS[run_name].synth(item["text"])
    return PROMPT_CACHE[key]

def synth_reference(run_name, item, source="ipa"):
    """Synthesises a held-out utterance either from its IPA label or from its raw Sinhala text."""
    key = (run_name, item["rel_path"], source)
    if key not in SYNTH_CACHE:
        model = MODELS[run_name]
        SYNTH_CACHE[key] = (model.synth_ipa(item["text"]) if source == "ipa"
                            else model.synth(item["sinhala"]))
    return SYNTH_CACHE[key]

def load_reference_audio(item):
    wav, sr = sf.read(item["audio_file"])
    return to_sr(wav, sr)

if not REF_PAIRS:
    skip_metric("objective", "MCD", "no run has ground-truth recordings available locally",
                "copy the dataset wavs/ folder to this machine")
else:
    mcd_rows = []
    for run_name in MODELS:
        for split, items in ref_sets(run_name):
            for item in items:
                ref = load_reference_audio(item)
                out = synth_reference(run_name, item)
                if not out["ok"]:
                    continue
                syn = to_sr(out["wav"], out["sr"])
                value, n_frames = mcd_dtw(ref, syn)
                mcd_rows.append({"run": run_name, "split": split, "rel_path": item["rel_path"],
                                 "mcd_db": value, "frames_scored": n_frames,
                                 "ref_s": len(ref) / COMMON_SR, "syn_s": len(syn) / COMMON_SR})
            print(f"  {run_name[:26]:26s} {split:8s} n={len(items):3d} "
                  f"MCD={np.nanmean([r['mcd_db'] for r in mcd_rows if r['run']==run_name and r['split']==split]):6.2f} dB")
    mcd_df = pd.DataFrame(mcd_rows)
    save_table(mcd_df, "11_mcd.csv")
    for (run_name, split), grp in mcd_df.groupby(["run", "split"]):
        label = "MCD" if split == "heldout" else "MCD_train_split"
        note = (f"std {grp['mcd_db'].std():.2f}; notebook-internal scale; "
                + ("held-out sentences" if split == "heldout"
                   else "sentences SEEN in training - generalisation reference only"))
        add_metric("objective", label, run_name, round(float(grp["mcd_db"].mean()), 2), "dB",
                   n=len(grp), note=note, higher_is_better=False)
    summary_mcd = mcd_df.groupby(["run", "split"])["mcd_db"].agg(["mean", "std", "min", "max", "count"]).round(2)
    print("\n", summary_mcd.to_string())
    for run_name, grp in mcd_df.groupby("run"):
        by_split = grp.groupby("split")["mcd_db"].mean()
        if {"heldout", "train"} <= set(by_split.index):
            gap = float(by_split["heldout"] - by_split["train"])
            add_metric("objective", "MCD_generalisation_gap", run_name, round(gap, 2), "dB",
                       n=len(grp), note="held-out minus training-split MCD; large positive = memorising",
                       higher_is_better=False)
            print(f"  {run_name[:34]:34s} generalisation gap {gap:+.2f} dB")

### 7.1 Checkpoint sweep — is `best_model.pth` actually the best one?

Coqui picks `best_model.pth` by validation **loss**, which is not the same thing as the audio
being better. This sweep loads every checkpoint in every run (skipping byte-identical duplicates)
and scores each one on the same held-out sentences with MCD, the duration ratio against the
recording, and the speaking rate. If a mid-training checkpoint wins, that is worth knowing before
shipping the "best" one.

Each checkpoint is ~1 GB, so this is the slowest cell in the notebook — set
`CONFIG["checkpoint_sweep"] = False` to skip it, or lower `CONFIG["n_sweep_pairs"]`.

In [ ]:
CONFIG.setdefault("checkpoint_sweep", True)
CONFIG.setdefault("n_sweep_pairs", 3)

sweep_rows = []
if not CONFIG["checkpoint_sweep"]:
    skip_metric("objective", "per-checkpoint quality sweep", "disabled in CONFIG['checkpoint_sweep']",
                "set it to True (expect a few minutes per checkpoint)")
elif not REF_PAIRS:
    skip_metric("objective", "per-checkpoint quality sweep",
                "needs held-out recordings to score each checkpoint against",
                "copy the dataset wavs/ folder to this machine")
else:
    seen_md5 = {}
    for run_name, run in RUNS.items():
        if run_name not in REF_PAIRS:
            print(f"  {run_name}: no reference pairs, skipping sweep")
            continue
        pairs = REF_PAIRS[run_name][: CONFIG["n_sweep_pairs"]]
        refs = [(item, load_reference_audio(item)) for item in pairs]
        for ckpt in run["checkpoints"]:
            digest = ckpt_df[(ckpt_df.run == run_name) & (ckpt_df.checkpoint == ckpt)]["md5"]
            digest = digest.iloc[0] if len(digest) else None
            if digest and digest in seen_md5:
                print(f"  {run_name[:22]:22s} {ckpt:22s} identical to {seen_md5[digest]} - skipped")
                continue
            if digest:
                seen_md5[digest] = f"{run_name[:12]}/{ckpt}"
            try:
                model = VoiceLKModel(run, checkpoint=ckpt, device=DEVICE)
            except Exception as exc:
                print(f"  {run_name[:22]:22s} {ckpt:22s} LOAD FAILED: {type(exc).__name__}")
                continue
            values, ratios, rates = [], [], []
            for item, ref in refs:
                out = model.synth_ipa(item["text"])
                if not out["ok"]:
                    continue
                syn = to_sr(out["wav"], out["sr"])
                values.append(mcd_dtw(ref, syn)[0])
                ratios.append(len(syn) / max(len(ref), 1))
                rates.append(out["n_tokens"] / out["audio_s"] if out["audio_s"] else float("nan"))
            step = ckpt_df[(ckpt_df.run == run_name) & (ckpt_df.checkpoint == ckpt)]["step"]
            sweep_rows.append({
                "run": run_name, "checkpoint": ckpt,
                "step": int(step.iloc[0]) if len(step) and pd.notna(step.iloc[0]) else _ckpt_step(ckpt),
                "is_primary": ckpt == run["primary_checkpoint"],
                "mcd_db": float(np.mean(values)) if values else float("nan"),
                "duration_ratio": float(np.mean(ratios)) if ratios else float("nan"),
                "tokens_per_second": float(np.mean(rates)) if rates else float("nan"),
                "n_pairs": len(values)})
            print(f"  {run_name[:22]:22s} {ckpt:22s} step={sweep_rows[-1]['step']:>7} "
                  f"MCD={sweep_rows[-1]['mcd_db']:6.2f} dB  len_ratio={sweep_rows[-1]['duration_ratio']:.3f}")
            del model

if sweep_rows:
    sweep_df = pd.DataFrame(sweep_rows).sort_values(["run", "step"])
    save_table(sweep_df, "11_checkpoint_sweep.csv")
    for run_name, grp in sweep_df.groupby("run"):
        valid = grp[grp["mcd_db"].notna()]
        if valid.empty:
            continue
        best = valid.loc[valid["mcd_db"].idxmin()]
        primary = valid[valid["is_primary"]]
        add_metric("objective", "best_checkpoint_by_MCD", run_name, best["checkpoint"], "",
                   n=int(best["step"]), note=f"MCD {best['mcd_db']:.2f} dB on {int(best['n_pairs'])} held-out pairs")
        if not primary.empty and primary.iloc[0]["checkpoint"] != best["checkpoint"]:
            print(f"\n  NOTE {run_name}: '{primary.iloc[0]['checkpoint']}' is the loss-selected checkpoint "
                  f"({primary.iloc[0]['mcd_db']:.2f} dB) but '{best['checkpoint']}' scores better "
                  f"({best['mcd_db']:.2f} dB)")

    fig, ax = plt.subplots(figsize=(9, 4.5))
    for run_name, grp in sweep_df.groupby("run"):
        grp = grp.sort_values("step")
        ax.plot(grp["step"], grp["mcd_db"], "o-", label=run_name[:24], alpha=0.85)
        prim = grp[grp["is_primary"]]
        if not prim.empty:
            ax.scatter(prim["step"], prim["mcd_db"], s=140, facecolors="none", edgecolors="crimson",
                       linewidths=1.6, label=f"{run_name[:16]} (selected)")
    ax.set_xlabel("training step"); ax.set_ylabel("MCD (dB, lower is better)")
    ax.set_title("Checkpoint quality over training — held-out sentences")
    ax.legend(fontsize=7); ax.grid(alpha=0.3)
    save_fig(fig, "11_checkpoint_sweep.png")
    plt.show()
    print(sweep_df.to_string(index=False))

---
## 8. Objective — F0 RMSE, pitch correlation, voicing agreement

F0 is tracked with `librosa.pyin` (65–400 Hz) on both the recording and the synthesis, the two
contours are DTW-aligned on log-F0, and three numbers are reported over the frames both tracks
call voiced — again for the held-out split and the training split separately:

* **F0 RMSE (Hz)** and **F0 RMSE (cents)** — cents is the perceptually meaningful one, since a
  20 Hz error matters far more at 100 Hz than at 300 Hz
* **pitch correlation** — Pearson r between the aligned contours (does the melody move together)
* **voicing agreement** — share of aligned frames where both tracks agree voiced/unvoiced

In [ ]:
def f0_metrics(ref, syn, sr=COMMON_SR, fmin=65.0, fmax=400.0):
    f0_r, v_r, _ = librosa.pyin(rms_norm(ref), fmin=fmin, fmax=fmax, sr=sr,
                                frame_length=1024, hop_length=256)
    f0_s, v_s, _ = librosa.pyin(rms_norm(syn), fmin=fmin, fmax=fmax, sr=sr,
                                frame_length=1024, hop_length=256)
    log_r = np.log(np.nan_to_num(f0_r, nan=1.0))[None, :]
    log_s = np.log(np.nan_to_num(f0_s, nan=1.0))[None, :]
    _, path = librosa.sequence.dtw(X=log_r, Y=log_s, metric="euclidean")
    ia, ib = path[:, 0], path[:, 1]
    voiced_agreement = float(np.mean(v_r[ia] == v_s[ib]))
    both = v_r[ia] & v_s[ib]
    if both.sum() < 5:
        return dict(f0_rmse_hz=float("nan"), f0_rmse_cents=float("nan"), f0_corr=float("nan"),
                    voicing_agreement=voiced_agreement, f0_bias_hz=float("nan"),
                    ref_mean_f0=float("nan"), syn_mean_f0=float("nan"), n_voiced=int(both.sum()))
    a, b = f0_r[ia][both], f0_s[ib][both]
    return dict(
        f0_rmse_hz=float(np.sqrt(np.mean((a - b) ** 2))),
        f0_rmse_cents=float(np.sqrt(np.mean((1200 * np.log2(b / a)) ** 2))),
        f0_corr=float(np.corrcoef(a, b)[0, 1]),
        voicing_agreement=voiced_agreement,
        f0_bias_hz=float(np.mean(b) - np.mean(a)),
        ref_mean_f0=float(np.mean(a)), syn_mean_f0=float(np.mean(b)), n_voiced=int(both.sum()))

if not REF_PAIRS:
    skip_metric("objective", "F0 RMSE / pitch correlation",
                "needs paired ground-truth recordings, none available locally",
                "copy the dataset wavs/ folder to this machine")
else:
    f0_rows = []
    for run_name in MODELS:
        for split, items in ref_sets(run_name):
            for item in items:
                out = synth_reference(run_name, item)
                if not out["ok"]:
                    continue
                row = {"run": run_name, "split": split, "rel_path": item["rel_path"]}
                row.update(f0_metrics(load_reference_audio(item), to_sr(out["wav"], out["sr"])))
                f0_rows.append(row)
            print(f"  {run_name[:26]:26s} {split:8s} n={len(items):3d} done")
    f0_df = pd.DataFrame(f0_rows)
    save_table(f0_df, "12_f0.csv")
    for (run_name, split), grp in f0_df.groupby(["run", "split"]):
        suffix = "" if split == "heldout" else "_train_split"
        note = "" if split == "heldout" else "sentences SEEN in training - reference only"
        add_metric("objective", "F0_RMSE" + suffix, run_name, round(float(grp["f0_rmse_hz"].mean()), 1),
                   "Hz", n=len(grp), note=note, higher_is_better=False)
        add_metric("objective", "F0_RMSE_cents" + suffix, run_name,
                   round(float(grp["f0_rmse_cents"].mean()), 1), "cents", n=len(grp), note=note,
                   higher_is_better=False)
        add_metric("objective", "pitch_correlation" + suffix, run_name,
                   round(float(grp["f0_corr"].mean()), 3), "r", n=len(grp), note=note, higher_is_better=True)
        add_metric("objective", "voicing_agreement" + suffix, run_name,
                   round(float(grp["voicing_agreement"].mean()), 3), "fraction", n=len(grp), note=note,
                   higher_is_better=True)
    print("\n", f0_df.groupby(["run", "split"])[["f0_rmse_hz", "f0_rmse_cents", "f0_corr",
                                                 "voicing_agreement"]].mean().round(3).to_string())

---
## 9. Objective — PESQ and STOI

Both metrics were designed for a degraded signal that is **sample-aligned** with its reference
(codecs, denoisers, transmission). A TTS system re-generates the utterance with its own timing, so
the pair is not aligned and the absolute values will be far below what the same metric reports for
enhancement work. They are kept because they are still a consistent *relative* yardstick across
these runs, and the caveat is written into the output file so nobody quotes them as absolute
quality scores.

Alignment used here: resample to 16 kHz → RMS-normalise → truncate both to the shorter length.
`pesq` has no prebuilt wheel for Python 3.14, so it usually reports as skipped on this machine.

In [ ]:
def align_for_intrusive(ref, syn):
    ref, syn = rms_norm(ref), rms_norm(syn)
    n = min(len(ref), len(syn))
    return ref[:n], syn[:n]

CAVEAT = "not sample-aligned (TTS re-times the utterance) - relative comparison only"

if not REF_PAIRS:
    for metric in ("PESQ", "STOI"):
        skip_metric("objective", metric, "needs paired ground-truth recordings, none available locally",
                    "copy the dataset wavs/ folder to this machine")
else:
    pesq_fn = stoi_fn = None
    if CAPS["pesq"]:
        from pesq import pesq as pesq_fn
    else:
        skip_metric("objective", "PESQ",
                    "the 'pesq' package is not installed (it is a C extension with no Python 3.14 wheel on PyPI)",
                    "install MS Visual C++ Build Tools then `pip install pesq`, or run this section on Python 3.11")
    if CAPS["pystoi"]:
        from pystoi import stoi as stoi_fn
    else:
        skip_metric("objective", "STOI/ESTOI", "the 'pystoi' package is not installed",
                    "`pip install pystoi` (pure Python, installs cleanly on 3.14)")

    intrusive_rows = []
    if pesq_fn or stoi_fn:
        for run_name, items in REF_PAIRS.items():
            for item in items:
                out = synth_reference(run_name, item)
                if not out["ok"]:
                    continue
                ref, syn = align_for_intrusive(load_reference_audio(item), to_sr(out["wav"], out["sr"]))
                row = {"run": run_name, "rel_path": item["rel_path"], "scored_s": len(ref) / COMMON_SR}
                if pesq_fn:
                    try:
                        row["pesq_wb"] = float(pesq_fn(COMMON_SR, ref, syn, "wb"))
                    except Exception as exc:
                        row["pesq_error"] = f"{type(exc).__name__}: {exc}"
                if stoi_fn:
                    try:
                        row["stoi"] = float(stoi_fn(ref, syn, COMMON_SR, extended=False))
                        row["estoi"] = float(stoi_fn(ref, syn, COMMON_SR, extended=True))
                    except Exception as exc:
                        row["stoi_error"] = f"{type(exc).__name__}: {exc}"
                intrusive_rows.append(row)

    if intrusive_rows:
        intrusive_df = pd.DataFrame(intrusive_rows)
        intrusive_df["caveat"] = CAVEAT
        save_table(intrusive_df, "13_pesq_stoi.csv")
        for run_name, grp in intrusive_df.groupby("run"):
            for col, label, unit in (("pesq_wb", "PESQ_wb", "MOS-LQO"),
                                     ("stoi", "STOI", "fraction"), ("estoi", "ESTOI", "fraction")):
                if col in grp and grp[col].notna().any():
                    add_metric("objective", label, run_name, round(float(grp[col].mean()), 3), unit,
                               n=int(grp[col].notna().sum()), note=CAVEAT, higher_is_better=True)
        print(intrusive_df.groupby("run")[[c for c in ("pesq_wb", "stoi", "estoi") if c in intrusive_df]]
              .mean().round(3).to_string())

---
## 10. Objective — ASR-based WER / CER

Synthesised audio is transcribed by a Sinhala ASR model and compared with the text that was fed
to the TTS. The number only means something next to a **calibration topline**: the same ASR run
over the *real recordings* of the same sentences. ASR on Sinhala is itself error-prone, so

* `wer_ground_truth` — the ASR's own error rate on human speech (the floor this test can reach)
* `wer_synth_from_ipa` — the acoustic model alone, fed the dataset's IPA label
* `wer_synth_from_text` — the deployed path: raw Sinhala text → normaliser → G2P → model

The gap between the second/third and the first is what is attributable to the TTS system; the gap
between the second and the third isolates front-end (normaliser + G2P) damage.

First run downloads ~1.2 GB for `SpideyDLK/wav2vec2-large-xls-r-300m-sinhala-low-LR-part1`.
Set `CONFIG["asr_model_id"]` to a different checkpoint (e.g. `Lingalingeswaran/whisper-small-sinhala`)
to swap it.

In [ ]:
import unicodedata, re as _re

def normalize_for_scoring(text):
    text = unicodedata.normalize("NFC", str(text))
    text = _re.sub(r"[^\w\s඀-෿]", " ", text)
    return _re.sub(r"\s+", " ", text).strip().lower()

def _levenshtein(a, b):
    if len(a) < len(b):
        a, b = b, a
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]

def wer(reference, hypothesis):
    ref, hyp = normalize_for_scoring(reference).split(), normalize_for_scoring(hypothesis).split()
    return float("nan") if not ref else _levenshtein(ref, hyp) / len(ref)

def cer(reference, hypothesis):
    ref, hyp = normalize_for_scoring(reference).replace(" ", ""), normalize_for_scoring(hypothesis).replace(" ", "")
    return float("nan") if not ref else _levenshtein(ref, hyp) / len(ref)

ASR = None
if not CONFIG["enable_asr"]:
    skip_metric("objective", "ASR WER/CER", "disabled in CONFIG['enable_asr']", "set it to True and re-run")
elif not CAPS["transformers"]:
    skip_metric("objective", "ASR WER/CER", "the 'transformers' package is not installed",
                "`pip install transformers` (then ~1.2 GB of ASR weights download on first use)")
else:
    try:
        from transformers import pipeline as hf_pipeline
        t0 = time.perf_counter()
        ASR = hf_pipeline("automatic-speech-recognition", model=CONFIG["asr_model_id"],
                          device=0 if DEVICE == "cuda" else -1)
        print(f"ASR loaded: {CONFIG['asr_model_id']} ({time.perf_counter() - t0:.1f}s)")
    except Exception as exc:
        skip_metric("objective", "ASR WER/CER", f"ASR model could not be loaded: {type(exc).__name__}: {exc}",
                    "check the internet connection / model id in CONFIG['asr_model_id']")

def transcribe(wav, sr=COMMON_SR):
    return ASR({"array": np.asarray(wav, dtype=np.float32), "sampling_rate": int(sr)})["text"]

In [ ]:
asr_rows = []
if ASR is not None and REF_PAIRS:
    for run_name, items in REF_PAIRS.items():
        for item in items:
            if not item["sinhala"]:
                continue
            reference = item["sinhala"]
            row = {"run": run_name, "rel_path": item["rel_path"], "reference": reference}

            hyp_gt = transcribe(load_reference_audio(item))
            row.update({"hyp_ground_truth": hyp_gt,
                        "wer_ground_truth": wer(reference, hyp_gt), "cer_ground_truth": cer(reference, hyp_gt)})

            out_ipa = synth_reference(run_name, item, "ipa")
            if out_ipa["ok"]:
                hyp = transcribe(to_sr(out_ipa["wav"], out_ipa["sr"]))
                row.update({"hyp_synth_from_ipa": hyp,
                            "wer_synth_from_ipa": wer(reference, hyp), "cer_synth_from_ipa": cer(reference, hyp)})

            out_txt = synth_reference(run_name, item, "text")
            if out_txt["ok"]:
                hyp = transcribe(to_sr(out_txt["wav"], out_txt["sr"]))
                row.update({"hyp_synth_from_text": hyp,
                            "wer_synth_from_text": wer(reference, hyp), "cer_synth_from_text": cer(reference, hyp)})
            asr_rows.append(row)
            print(f"  {run_name[:22]:22s} {item['rel_path'][:22]:22s} "
                  f"WER gt={row.get('wer_ground_truth', float('nan')):.3f} "
                  f"ipa={row.get('wer_synth_from_ipa', float('nan')):.3f} "
                  f"text={row.get('wer_synth_from_text', float('nan')):.3f}")

if ASR is not None:
    # same ASR over the hand-written prompt set (no recordings exist for these, so no topline)
    for run_name, model in MODELS.items():
        for item in EVAL_SENTENCES:
            if item["category"] in ("symbol", "url", "number"):
                continue        # the normaliser rewrites these, so the prompt is not the spoken text
            out = model.synth(item["text"])
            if not out["ok"]:
                continue
            hyp = transcribe(to_sr(out["wav"], out["sr"]))
            asr_rows.append({"run": run_name, "rel_path": f"prompt:{item['id']}", "category": item["category"],
                             "reference": item["text"], "hyp_synth_from_text": hyp,
                             "wer_synth_from_text": wer(item["text"], hyp),
                             "cer_synth_from_text": cer(item["text"], hyp)})

if asr_rows:
    asr_df = pd.DataFrame(asr_rows)
    asr_df["asr_model"] = CONFIG["asr_model_id"]
    save_table(asr_df, "15_asr_wer.csv")
    held = asr_df[~asr_df["rel_path"].str.startswith("prompt:")]
    for run_name, grp in held.groupby("run"):
        for col, label in (("wer_ground_truth", "WER_asr_topline_on_real_speech"),
                           ("wer_synth_from_ipa", "WER_synth_from_ipa"),
                           ("wer_synth_from_text", "WER_synth_end_to_end"),
                           ("cer_synth_from_text", "CER_synth_end_to_end")):
            if col in grp and grp[col].notna().any():
                add_metric("objective", label, run_name, round(float(grp[col].mean()), 3), "rate",
                           n=int(grp[col].notna().sum()),
                           note=f"ASR={CONFIG['asr_model_id']}", higher_is_better=False)
        if {"wer_ground_truth", "wer_synth_from_text"} <= set(grp.columns):
            delta = float(grp["wer_synth_from_text"].mean() - grp["wer_ground_truth"].mean())
            add_metric("objective", "WER_gap_vs_real_speech", run_name, round(delta, 3), "rate",
                       n=len(grp), note="synthetic WER minus ASR topline on the same sentences",
                       higher_is_better=False)
    print("\n", held.groupby("run")[[c for c in ("wer_ground_truth", "wer_synth_from_ipa",
          "wer_synth_from_text") if c in held]].mean().round(3).to_string())

---
## 11. Objective — Neural MOS prediction (UTMOS)

UTMOS22 (`tarepan/SpeechMOS` via `torch.hub`) predicts a 1–5 naturalness MOS without listeners.
It was trained on English/Japanese VCC and BC systems, so for Sinhala treat it as a *ranking*
signal, not as a calibrated MOS. Scoring the real recordings alongside the synthesis gives the
ceiling this predictor assigns to this speaker and recording chain.

In [ ]:
utmos = None
if not CONFIG["enable_utmos"]:
    skip_metric("objective", "Neural MOS (UTMOS)", "disabled in CONFIG['enable_utmos']", "set it to True")
else:
    try:
        utmos = torch.hub.load("tarepan/SpeechMOS:v1.2.0", "utmos22_strong", trust_repo=True)
        utmos.eval()
        print("UTMOS loaded")
    except Exception as exc:
        skip_metric("objective", "Neural MOS (UTMOS)",
                    f"torch.hub could not fetch the predictor: {type(exc).__name__}: {exc}",
                    "needs internet access on first use; alternatives are NISQA or MOSNet")

if utmos is not None:
    def utmos_score(wav, sr=COMMON_SR):
        with torch.no_grad():
            return float(utmos(torch.from_numpy(np.asarray(wav, dtype=np.float32)).unsqueeze(0), int(sr)))

    utmos_rows = []
    for run_name, model in MODELS.items():
        for item in EVAL_SENTENCES:
            out = synth_prompt(run_name, item)
            if out["ok"] and out["audio_s"] > 0.3:
                utmos_rows.append({"run": run_name, "source": "synth_prompt", "id": item["id"],
                                   "category": item["category"], "utmos": utmos_score(to_sr(out["wav"], out["sr"]))})
    for run_name, items in REF_PAIRS.items():
        for item in items:
            out = synth_reference(run_name, item)
            if out["ok"]:
                utmos_rows.append({"run": run_name, "source": "synth_heldout", "id": item["rel_path"],
                                   "category": "heldout", "utmos": utmos_score(to_sr(out["wav"], out["sr"]))})
            utmos_rows.append({"run": "GROUND_TRUTH", "source": "recording", "id": item["rel_path"],
                               "category": "heldout", "utmos": utmos_score(load_reference_audio(item))})

    utmos_df = pd.DataFrame(utmos_rows).drop_duplicates(subset=["run", "source", "id"])
    save_table(utmos_df, "16_utmos.csv")
    for run_name, grp in utmos_df.groupby("run"):
        add_metric("objective", "UTMOS_predicted_MOS", run_name, round(float(grp["utmos"].mean()), 3), "MOS 1-5",
                   n=len(grp), note="UTMOS22-strong; trained on English/Japanese, use as a ranking signal",
                   higher_is_better=True)
    print(utmos_df.groupby(["run", "source"])["utmos"].agg(["mean", "std", "count"]).round(3).to_string())

    fig, ax = plt.subplots(figsize=(8, 4))
    groups = list(utmos_df.groupby("run"))
    ax.boxplot([g["utmos"] for _, g in groups], tick_labels=[n[:20] for n, _ in groups])
    ax.set_ylabel("UTMOS (1-5)"); ax.set_title("Predicted naturalness — synthesis vs real recordings")
    ax.tick_params(axis="x", labelsize=7, rotation=15)
    save_fig(fig, "16_utmos.png")
    plt.show()

---
## 12. Objective — Speaker similarity and speaker consistency

Two different questions:

* **similarity** — does a synthesised utterance sound like *this speaker*? (cosine between the
  embedding of the synthesis and of that speaker's real recording)
* **consistency** — does the model keep one identity across utterances? (mean pairwise cosine
  between embeddings of different synthesised utterances from the same model)

With `speechbrain` installed both use ECAPA-TDNN x-vectors. Without it, a fallback descriptor is
used — long-term average spectrum + F0 statistics, mean-centred over the evaluation set. The
fallback is reported as `proxy` and is *not* a speaker-verification score: it reacts to recording
channel and content as well as to voice identity. A real-speaker cosine (recording vs recording)
is always computed alongside so there is a calibration point.

In [ ]:
encoder, EMB_KIND = None, "proxy"
if CONFIG["enable_speaker_encoder"] and CAPS["speechbrain"]:
    try:
        from speechbrain.inference.speaker import EncoderClassifier
        encoder = EncoderClassifier.from_hparams(
            source=CONFIG["speaker_encoder_id"],
            savedir=os.path.join(OUT_DIR, "_speechbrain_ecapa"),
            run_opts={"device": DEVICE})
        EMB_KIND = "ecapa"
        print("speaker encoder: ECAPA-TDNN")
    except Exception as exc:
        skip_metric("objective", "Speaker similarity (ECAPA)",
                    f"speechbrain model could not be loaded: {type(exc).__name__}: {exc}",
                    "check internet access; falling back to the spectral proxy")
elif not CAPS["speechbrain"]:
    skip_metric("objective", "Speaker similarity (ECAPA x-vector)",
                "the 'speechbrain' package is not installed, so a spectral proxy is used instead",
                "`pip install speechbrain` for a real speaker-verification embedding")

def embed(wav, sr=COMMON_SR):
    if encoder is not None:
        with torch.no_grad():
            vec = encoder.encode_batch(torch.from_numpy(np.asarray(wav, dtype=np.float32)).unsqueeze(0))
        return vec.squeeze().detach().cpu().numpy()
    wav = rms_norm(wav)
    ltas = np.log(librosa.feature.melspectrogram(y=wav, sr=sr, n_mels=40, n_fft=1024,
                                                 hop_length=256, power=2.0).mean(axis=1) + 1e-10)
    f0, voiced, _ = librosa.pyin(wav, fmin=65.0, fmax=400.0, sr=sr, frame_length=1024, hop_length=256)
    f0v = f0[voiced] if voiced.any() else np.array([0.0])
    return np.concatenate([ltas, [np.nanmean(f0v), np.nanstd(f0v),
                                  float(librosa.feature.spectral_centroid(y=wav, sr=sr).mean())/1000.0]])

def cosine(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))

if not REF_PAIRS:
    skip_metric("objective", "Speaker similarity", "needs paired ground-truth recordings",
                "copy the dataset wavs/ folder to this machine")
else:
    records, gt_done = [], set()
    for run_name, items in REF_PAIRS.items():
        for item in items:
            out = synth_reference(run_name, item)
            if not out["ok"]:
                continue
            records.append({"run": run_name, "rel_path": item["rel_path"], "kind": "synth",
                            "emb": embed(to_sr(out["wav"], out["sr"]))})
            # runs that share a dataset share held-out items - embed each recording once only,
            # otherwise identical pairs inflate the real-speaker calibration below
            if item["rel_path"] not in gt_done:
                gt_done.add(item["rel_path"])
                records.append({"run": "GROUND_TRUTH", "rel_path": item["rel_path"], "kind": "recording",
                                "emb": embed(load_reference_audio(item))})

    embs = np.stack([r["emb"] for r in records])
    if EMB_KIND == "proxy":                       # centring makes the proxy discriminative at all
        embs = embs - embs.mean(axis=0)
    for rec, vec in zip(records, embs):
        rec["emb"] = vec

    gt_by_path = {r["rel_path"]: r["emb"] for r in records if r["kind"] == "recording"}
    sim_rows = []
    for rec in records:
        if rec["kind"] != "synth":
            continue
        sim_rows.append({"run": rec["run"], "rel_path": rec["rel_path"], "embedding": EMB_KIND,
                         "similarity_to_own_recording": cosine(rec["emb"], gt_by_path[rec["rel_path"]])})

    def mean_pairwise(vectors):
        vals = [cosine(vectors[i], vectors[j]) for i in range(len(vectors)) for j in range(i + 1, len(vectors))]
        return float(np.mean(vals)) if vals else float("nan")

    consistency_rows = []
    for run_name in MODELS:
        vecs = [r["emb"] for r in records if r["run"] == run_name]
        if len(vecs) > 1:
            consistency_rows.append({"run": run_name, "embedding": EMB_KIND,
                                     "self_consistency": mean_pairwise(vecs), "n": len(vecs)})
    gt_vecs = [r["emb"] for r in records if r["kind"] == "recording"]
    if len(gt_vecs) > 1:
        consistency_rows.append({"run": "GROUND_TRUTH", "embedding": EMB_KIND,
                                 "self_consistency": mean_pairwise(gt_vecs), "n": len(gt_vecs)})

    spk_df = pd.DataFrame(sim_rows)
    cons_df = pd.DataFrame(consistency_rows)
    save_table(spk_df, "17_speaker_similarity.csv")
    save_table(cons_df, "17_speaker_consistency.csv")
    note = ("ECAPA-TDNN x-vector cosine" if EMB_KIND == "ecapa"
            else "spectral proxy (LTAS + F0 stats), content-sensitive - not a verification score")
    for run_name, grp in spk_df.groupby("run"):
        add_metric("objective", "speaker_similarity_to_reference", run_name,
                   round(float(grp["similarity_to_own_recording"].mean()), 3), "cosine",
                   n=len(grp), note=note, higher_is_better=True)
    for row in consistency_rows:
        add_metric("component", "speaker_consistency", row["run"], round(row["self_consistency"], 3), "cosine",
                   n=int(row["n"]), note=note + " (GROUND_TRUTH row is the real-speaker calibration)",
                   higher_is_better=True)
    print(spk_df.groupby("run")["similarity_to_own_recording"].agg(["mean", "std", "count"]).round(3).to_string())
    print("\n", cons_df.to_string(index=False))
    if EMB_KIND == "proxy":
        print("\nNOTE: these are mean-centred proxy cosines, so they are only meaningful *relative* to the "
              "GROUND_TRUTH row, and centring needs a reasonable sample - with fewer than ~6 utterances "
              "per system the values (including negative ones) are noise. Install speechbrain for real "
              "ECAPA x-vectors, or raise CONFIG['n_ref_pairs'].")

---
## 13. Objective — prosody and duration alignment

VITS predicts a duration per input token and the stochastic duration predictor turns that into the
output length, so three things are checked:

* **per-token duration distribution** — a predictor that has collapsed emits ~1 frame for every
  token, which sounds rushed and flat
* **utterance length agreement** — synthesised duration vs the real recording of the same
  sentence (ratio, and correlation across the held-out set)
* **timing drift** — how far the DTW path between synthesis and recording departs from the
  diagonal, i.e. whether the two drift apart inside the utterance even when the totals match

A phoneme-level comparison against forced-alignment ground truth is **not** computed: that needs a
forced aligner with a Sinhala acoustic model (e.g. Montreal Forced Aligner), which is not
installed here — recorded as a skip below.

In [ ]:
skip_metric("component", "phoneme-level duration vs forced alignment",
            "no forced aligner with a Sinhala acoustic model is available on this machine",
            "train/obtain an MFA Sinhala acoustic model, align the corpus, then compare per-phoneme "
            "durations against the VITS duration predictor")

def dtw_timing_drift(ref, syn):
    """RMS deviation (in frames) of the DTW path from the diagonal, length-normalised."""
    a, _ = mel_cepstrum(ref)
    b, _ = mel_cepstrum(syn)
    _, path = librosa.sequence.dtw(X=a.T, Y=b.T, metric="euclidean")
    ia, ib = path[::-1, 0], path[::-1, 1]
    expected = ia * (len(b) - 1) / max(len(a) - 1, 1)
    return float(np.sqrt(np.mean((ib - expected) ** 2)))

dur_rows = []
for run_name, model in MODELS.items():
    for item in EVAL_SENTENCES:
        out = synth_prompt(run_name, item)
        if not out["ok"] or out["durations"].size == 0:
            continue
        d = out["durations"]
        dur_rows.append({"run": run_name, "id": item["id"], "category": item["category"],
                         "source": "prompt", "tokens": out["n_tokens"],
                         "dur_mean_frames": float(d.mean()), "dur_std_frames": float(d.std()),
                         "dur_min": float(d.min()), "dur_max": float(d.max()),
                         "frames_at_minimum_pct": float((d <= 1.0).mean() * 100),
                         "audio_s": out["audio_s"],
                         "chars_per_second": len(item["text"]) / out["audio_s"] if out["audio_s"] else float("nan")})

for run_name, items in REF_PAIRS.items():
    for item in items:
        out = synth_reference(run_name, item)
        if not out["ok"]:
            continue
        ref = load_reference_audio(item)
        syn = to_sr(out["wav"], out["sr"])
        d = out["durations"]
        dur_rows.append({"run": run_name, "id": item["rel_path"], "category": "heldout", "source": "heldout",
                         "tokens": out["n_tokens"],
                         "dur_mean_frames": float(d.mean()) if d.size else float("nan"),
                         "dur_std_frames": float(d.std()) if d.size else float("nan"),
                         "frames_at_minimum_pct": float((d <= 1.0).mean() * 100) if d.size else float("nan"),
                         "audio_s": len(syn) / COMMON_SR, "reference_s": len(ref) / COMMON_SR,
                         "duration_ratio": (len(syn) / len(ref)) if len(ref) else float("nan"),
                         "timing_drift_frames": dtw_timing_drift(ref, syn)})

dur_df = pd.DataFrame(dur_rows)
save_table(dur_df, "18_duration_prosody.csv")

for run_name, grp in dur_df.groupby("run"):
    add_metric("component", "predicted_duration_mean", run_name, round(float(grp["dur_mean_frames"].mean()), 2),
               "frames/token", n=len(grp),
               note="~1.0 means the duration predictor has collapsed to its minimum", higher_is_better=None)
    add_metric("component", "tokens_at_minimum_duration", run_name,
               round(float(grp["frames_at_minimum_pct"].mean()), 1), "%", n=len(grp), higher_is_better=False)
    held = grp[grp["source"] == "heldout"]
    if not held.empty and held["duration_ratio"].notna().any():
        add_metric("component", "duration_ratio_vs_recording", run_name,
                   round(float(held["duration_ratio"].mean()), 3), "ratio", n=len(held),
                   note="1.0 = same length as the real recording", higher_is_better=None)
        add_metric("component", "timing_drift", run_name, round(float(held["timing_drift_frames"].mean()), 1),
                   "frames", n=len(held), note="RMS deviation of the DTW path from the diagonal",
                   higher_is_better=False)

print(dur_df.groupby("run")[["dur_mean_frames", "frames_at_minimum_pct"]].mean().round(2).to_string())

# per-token duration profile for one sentence, per model
for run_name, model in MODELS.items():
    out = synth_prompt(run_name, EVAL_SENTENCES[0])
    if not out["ok"] or out["durations"].size == 0:
        continue
    d = out["durations"]
    chars = list(out["ipa"].replace(" ", "_"))[:len(d)]
    fig, ax = plt.subplots(figsize=(max(9, len(chars) * 0.22), 3))
    ax.bar(range(len(d)), d, color="#2a6f97")
    ax.set_xticks(range(len(chars)))
    ax.set_xticklabels(chars, rotation=90, fontsize=6)
    ax.set_ylabel("predicted frames")
    ax.set_title(f"{run_name} — per-token predicted duration (gen01)")
    save_fig(fig, f"18_durations_{run_name[:28]}.png")
    plt.show()

held_all = dur_df[(dur_df["source"] == "heldout") & dur_df["duration_ratio"].notna()]
if not held_all.empty:
    fig, ax = plt.subplots(figsize=(6, 5))
    for run_name, grp in held_all.groupby("run"):
        ax.scatter(grp["reference_s"], grp["audio_s"], s=28, alpha=0.8, label=run_name[:22])
    lims = [0, held_all[["reference_s", "audio_s"]].to_numpy().max() * 1.1]
    ax.plot(lims, lims, "--", color="grey", linewidth=1, label="perfect length match")
    ax.set_xlabel("recording duration (s)"); ax.set_ylabel("synthesis duration (s)")
    ax.set_title("Utterance length: synthesis vs recording"); ax.legend(fontsize=7)
    save_fig(fig, "18_duration_alignment.png")
    plt.show()

---
## 14. Component — G2P accuracy and IPA→vocabulary coverage

Three separate things are measured, because "G2P accuracy" hides all three:

1. **Lexicon fidelity** — for every entry in `lexicon.json`, does the full pipeline actually emit
   the IPA the lexicon specifies? A miss means the override never fired (tokenisation,
   normalisation or casing ate it), which is a silent mispronunciation.
2. **Phoneme error rate** — character-level edit distance against the lexicon IPA, so near-misses
   are distinguishable from complete failures.
3. **Vocabulary coverage** — the share of produced IPA characters that the trained model can
   actually represent. Coqui's tokenizer *discards* unknown characters without raising, so a
   symbol the front-end emits but the model never saw (`ə`, `ˈ`, …) is dropped silently. This is
   measured per model, since the two dataset families were trained with different vocabularies.

A gold set verified by a phonetician would be needed to call any of this "G2P accuracy" in the
linguistic sense — the lexicon is the only machine-checkable reference available, and it is the
same file the pipeline reads, so item 1 measures *application*, not *correctness*, of the lexicon.

In [ ]:
skip_metric("component", "G2P accuracy vs a human-verified phonetic gold set",
            "no human-transcribed IPA reference exists for this corpus; the lexicon is the only "
            "machine-checkable reference and the pipeline reads that same file",
            "have a phonetician transcribe ~200 sampled words (Sinhala, English loanwords, acronyms) "
            "and score the pipeline against that set")

with open(os.path.join(VOICELK_ML, "model_engine", "lexicon.json"), encoding="utf-8") as fh:
    LEXICON = json.load(fh)
EN_LEX, SI_LEX = LEXICON.get("en_lexicon", {}), LEXICON.get("si_lexicon", {})
print(f"lexicon: {len(EN_LEX)} English entries, {len(SI_LEX)} Sinhala entries "
      f"(schema {LEXICON.get('metadata', {}).get('schema_version')})")

g2p_rows = []
for lex_name, lex in (("en_lexicon", EN_LEX), ("si_lexicon", SI_LEX)):
    for word, entry in lex.items():
        expected = (entry.get("ipa") or "").strip()
        if not expected:
            continue
        try:
            produced = TEXT_PIPELINE.process(word)["ipa_sequence"].strip()
            error = ""
        except Exception as exc:
            produced, error = "", f"{type(exc).__name__}: {exc}"
        distance = _levenshtein(list(expected), list(produced))
        g2p_rows.append({"lexicon": lex_name, "word": word, "expected_ipa": expected,
                         "produced_ipa": produced, "exact_match": produced == expected,
                         "per": distance / max(len(expected), 1), "error": error})

g2p_df = pd.DataFrame(g2p_rows)
save_table(g2p_df, "20_g2p_lexicon_fidelity.csv")
for lex_name, grp in g2p_df.groupby("lexicon"):
    acc, per = float(grp["exact_match"].mean()), float(grp["per"].mean())
    add_metric("component", f"lexicon_fidelity[{lex_name}]", "front-end (model-independent)",
               round(acc * 100, 1), "%", n=len(grp),
               note="share of lexicon entries the pipeline reproduces exactly", higher_is_better=True)
    add_metric("component", f"phoneme_error_rate[{lex_name}]", "front-end (model-independent)",
               round(per, 3), "PER", n=len(grp), higher_is_better=False)
    print(f"  {lex_name}: exact {acc*100:5.1f}%  PER {per:.3f}  ({len(grp)} entries)")

worst = g2p_df[~g2p_df["exact_match"]].sort_values("per", ascending=False).head(25)
save_table(worst, "20_g2p_worst_mismatches.csv")
print("\nworst lexicon mismatches:")
print(worst[["lexicon", "word", "expected_ipa", "produced_ipa", "per"]].head(12).to_string(index=False))

In [ ]:
# IPA -> model-vocabulary coverage: which characters does the tokenizer silently drop?
corpus_texts = [item["text"] for item in EVAL_SENTENCES]
corpus_ipa = []
for text in corpus_texts:
    try:
        corpus_ipa.append(TEXT_PIPELINE.process(text)["ipa_sequence"])
    except Exception:
        corpus_ipa.append("")

cov_rows, drop_rows = [], []
for run_name, model in MODELS.items():
    total = dropped = 0
    counter = Counter()
    for ipa in corpus_ipa:
        for ch in ipa:
            total += 1
            if model.vocab and ch not in model.vocab:
                dropped += 1
                counter[ch] += 1
    cov_rows.append({"run": run_name, "vocab_size": len(model.vocab), "chars_seen": total,
                     "chars_dropped": dropped,
                     "coverage_pct": 100 * (1 - dropped / total) if total else float("nan"),
                     "distinct_dropped": len(counter)})
    for ch, n in counter.most_common():
        drop_rows.append({"run": run_name, "char": ch, "unicode": f"U+{ord(ch):04X}",
                          "name": unicodedata.name(ch, "?"), "count": n})

cov_df, drop_df = pd.DataFrame(cov_rows), pd.DataFrame(drop_rows)
save_table(cov_df, "20_ipa_vocab_coverage.csv")
if not drop_df.empty:
    save_table(drop_df, "20_ipa_dropped_chars.csv")
for row in cov_rows:
    add_metric("component", "ipa_vocab_coverage", row["run"], round(row["coverage_pct"], 2), "%",
               n=row["chars_seen"],
               note=f"{row['chars_dropped']} chars silently discarded by the tokenizer "
                    f"({row['distinct_dropped']} distinct)", higher_is_better=True)
print(cov_df.to_string(index=False))
if not drop_df.empty:
    print("\nmost-dropped characters:")
    print(drop_df.sort_values("count", ascending=False).head(12).to_string(index=False))

---
## 15. Component — language-routing (code-switch) accuracy

`CodeSwitchedG2P` routes each token by regex: pure ASCII letters → English G2P, pure Sinhala block
→ Sinhala rules, anything else passes through untouched. The labelled probe below checks that the
router sends each token where it belongs, and — just as important — checks the **normaliser
first**, because digits, symbols and URLs are rewritten into Sinhala words before the router ever
sees them. A token that survives normalisation as a digit reaches the router as "other" and is
emitted unspoken.

In [ ]:
import re as _re2
SINHALA_RE = _re2.compile(r"^[඀-෿‍]+$")
ENGLISH_RE = _re2.compile(r"^[a-zA-Z]+$")

def route_token(token):
    if ENGLISH_RE.match(token):
        return "english"
    if SINHALA_RE.match(token):
        return "sinhala"
    return "other"

ROUTING_PROBE = [
    ("පරිගණකය", "sinhala"), ("තාක්ෂණය", "sinhala"), ("ඩිවයිස්", "sinhala"), ("මෘදුකාංග", "sinhala"),
    ("ක්‍රියාවලිය", "sinhala"), ("විද්‍යුත්", "sinhala"),
    ("computer", "english"), ("device", "english"), ("software", "english"), ("Sinhala", "english"),
    ("RAM", "english"), ("CPU", "english"), ("ICT", "english"), ("WFH", "english"),
    (".", "other"), (",", "other"), ("?", "other"), ("!", "other"),
    ("2026", "other"), ("15", "other"), ("COVID-19", "other"), ("test@example.com", "other"),
]

routing_rows = [{"token": tok, "expected": exp, "routed": route_token(tok),
                 "correct": route_token(tok) == exp} for tok, exp in ROUTING_PROBE]
routing_df = pd.DataFrame(routing_rows)
accuracy = float(routing_df["correct"].mean())
save_table(routing_df, "21_language_routing_tokens.csv")
add_metric("component", "language_routing_accuracy", "front-end (model-independent)",
           round(accuracy * 100, 1), "%", n=len(routing_df),
           note="token-level router probe (Sinhala / English / other)", higher_is_better=True)
print(f"token routing accuracy: {accuracy*100:.1f}%  ({int(routing_df['correct'].sum())}/{len(routing_df)})")
print(pd.crosstab(routing_df["expected"], routing_df["routed"]).to_string())

# sentence level: what does the router actually see after normalisation?
sent_rows = []
for item in EVAL_SENTENCES:
    normalized = TEXT_PIPELINE.normalizer.normalize(item["text"])
    tokens = TEXT_PIPELINE.g2p_engine.tokenizer.tokenize(normalized)
    counts = Counter(route_token(t) for t in tokens)
    leftover_digits = [t for t in tokens if any(ch.isdigit() for ch in t)]
    sent_rows.append({"id": item["id"], "category": item["category"], "tokens": len(tokens),
                      "sinhala": counts["sinhala"], "english": counts["english"], "other": counts["other"],
                      "unspoken_digit_tokens": len(leftover_digits),
                      "leftover": " ".join(leftover_digits)[:60]})
sent_df = pd.DataFrame(sent_rows)
save_table(sent_df, "21_language_routing_sentences.csv")
unspoken = int(sent_df["unspoken_digit_tokens"].sum())
add_metric("component", "digits_surviving_normalisation", "front-end (model-independent)", unspoken,
           "tokens", n=int(sent_df["tokens"].sum()),
           note="digit tokens still present after normalisation - these reach the model unspoken",
           higher_is_better=False)
print("\n", sent_df.to_string(index=False))

---
## 16. Component — lexicon coverage rate

How much of the real corpus vocabulary the ICT lexicon actually covers. Low coverage is not a bug
by itself — the rule-based fallback handles ordinary Sinhala — but every uncovered English
loanword or acronym is a pronunciation the lexicon is not controlling.

In [ ]:
def token_language(token):
    return route_token(token)

corpus_sources = {"eval_prompts": [item["text"] for item in EVAL_SENTENCES]}
if ORTHO:
    corpus_sources["dataset_transcripts"] = [v["sinhala"] for v in ORTHO.values() if v["sinhala"]]
    corpus_sources["dataset_code_switch"] = [v["code_switch"] for v in ORTHO.values() if v["code_switch"]]

coverage_rows, oov_rows = [], []
for source_name, texts in corpus_sources.items():
    counters = {"english": Counter(), "sinhala": Counter()}
    for text in texts:
        normalized = TEXT_PIPELINE.normalizer.normalize(text)
        for token in TEXT_PIPELINE.g2p_engine.tokenizer.tokenize(normalized):
            lang = token_language(token)
            if lang in counters:
                counters[lang][token] += 1
    for lang, counter in counters.items():
        lex = EN_LEX if lang == "english" else SI_LEX
        def covered(tok):
            return (tok.lower() in lex) if lang == "english" else (tok in lex)
        tokens_total = sum(counter.values())
        tokens_hit = sum(n for tok, n in counter.items() if covered(tok))
        types_hit = sum(1 for tok in counter if covered(tok))
        coverage_rows.append({
            "source": source_name, "language": lang,
            "token_count": tokens_total, "type_count": len(counter),
            "token_coverage_pct": 100 * tokens_hit / tokens_total if tokens_total else float("nan"),
            "type_coverage_pct": 100 * types_hit / len(counter) if counter else float("nan")})
        for tok, n in counter.most_common():
            if not covered(tok):
                oov_rows.append({"source": source_name, "language": lang, "token": tok, "count": n})

coverage_df = pd.DataFrame(coverage_rows)
oov_df = pd.DataFrame(oov_rows)
save_table(coverage_df, "22_lexicon_coverage.csv")
if not oov_df.empty:
    save_table(oov_df.sort_values("count", ascending=False).head(300), "22_lexicon_oov_top300.csv")
for row in coverage_rows:
    add_metric("component", f"lexicon_coverage[{row['source']}/{row['language']}]",
               "front-end (model-independent)", round(row["token_coverage_pct"], 2), "% of tokens",
               n=row["token_count"], note=f"type coverage {row['type_coverage_pct']:.1f}%",
               higher_is_better=True)
print(coverage_df.round(2).to_string(index=False))
if not oov_df.empty:
    print("\nmost frequent out-of-lexicon English tokens:")
    print(oov_df[oov_df.language == "english"].sort_values("count", ascending=False)
          .head(15).to_string(index=False))

---
## 17. System — end-to-end latency under load

Concurrent requests through a thread pool, which is how the API in `api/` would serve it. Because
the model runs in-process, concurrency is bounded by the GIL and by torch's intra-op threads — the
point of this test is the **shape** of the degradation (p95 and throughput as concurrency rises),
not an absolute capacity number for a production deployment.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

def timed_request(model, text):
    t0 = time.perf_counter()
    out = model.synth(text)
    return {"latency_s": time.perf_counter() - t0, "audio_s": out["audio_s"], "ok": out["ok"]}

load_rows = []
prompts = [item["text"] for item in EVAL_SENTENCES]
for run_name, model in MODELS.items():
    model.synth(prompts[0])                                     # warm-up
    for workers in CONFIG["load_concurrency"]:
        batch = [prompts[i % len(prompts)] for i in range(CONFIG["load_requests"])]
        start = time.perf_counter()
        with ThreadPoolExecutor(max_workers=workers) as pool:
            results = list(pool.map(lambda t: timed_request(model, t), batch))
        wall = time.perf_counter() - start
        lat = np.array([r["latency_s"] for r in results]) * 1000
        audio_total = sum(r["audio_s"] for r in results)
        load_rows.append({
            "run": run_name, "concurrency": workers, "requests": len(batch),
            "wall_s": round(wall, 2),
            "throughput_req_per_s": round(len(batch) / wall, 3),
            "audio_generated_s": round(audio_total, 1),
            "throughput_audio_x_realtime": round(audio_total / wall, 2),
            "p50_ms": round(float(np.percentile(lat, 50)), 1),
            "p95_ms": round(float(np.percentile(lat, 95)), 1),
            "max_ms": round(float(lat.max()), 1),
            "failures": int(sum(not r["ok"] for r in results))})
        print(f"  {run_name[:24]:24s} concurrency={workers}  {load_rows[-1]['throughput_req_per_s']:5.2f} req/s  "
              f"p50={load_rows[-1]['p50_ms']:7.1f} ms  p95={load_rows[-1]['p95_ms']:7.1f} ms")

load_df = pd.DataFrame(load_rows)
save_table(load_df, "30_load_latency.csv")
for run_name, grp in load_df.groupby("run"):
    best = grp.loc[grp["throughput_req_per_s"].idxmax()]
    add_metric("system", "peak_throughput", run_name, float(best["throughput_req_per_s"]), "req/s",
               n=int(best["concurrency"]), note=f"at concurrency {int(best['concurrency'])} on {DEVICE}",
               higher_is_better=True)
    worst = grp.loc[grp["concurrency"].idxmax()]
    add_metric("system", "latency_p95_under_load", run_name, float(worst["p95_ms"]), "ms",
               n=int(worst["concurrency"]), note=f"at concurrency {int(worst['concurrency'])}",
               higher_is_better=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for run_name, grp in load_df.groupby("run"):
    axes[0].plot(grp["concurrency"], grp["throughput_req_per_s"], "o-", label=run_name[:20])
    axes[1].plot(grp["concurrency"], grp["p95_ms"], "o-", label=run_name[:20])
axes[0].set_xlabel("concurrent workers"); axes[0].set_ylabel("requests / s"); axes[0].set_title("Throughput")
axes[1].set_xlabel("concurrent workers"); axes[1].set_ylabel("p95 latency (ms)"); axes[1].set_title("Tail latency")
for ax in axes:
    ax.legend(fontsize=7); ax.grid(alpha=0.3)
save_fig(fig, "30_load_latency.png")
plt.show()
load_df

---
## 18. System — robustness and edge cases

Each probe goes through the full deployed path (normalise → G2P → tokenize → synthesise) and is
flagged when the output is implausible rather than merely different: empty or near-empty audio for
speakable text, a speaking rate outside 3–40 characters/second, near-silence, or clipping.
Front-end crashes are caught and recorded rather than stopping the notebook.

Inputs with nothing speakable in them (empty string, whitespace, emoji-only) are expected to
produce no audio, so they are marked `empty_output_expected` and not counted as failures — those
would otherwise show up as phantom defects in the error taxonomy.

In [ ]:
EDGE_CASES = [
    ("empty",          ""),
    ("whitespace",     "   "),
    ("punct_only",     "...!!!???"),
    ("emoji_only",     "😀🎉🔥"),
    ("emoji_mixed",    "සුබ උදෑසනක් 😀 යාළුවනේ!"),
    ("repeated_char",  "අඅඅඅඅඅඅඅඅඅඅඅඅඅඅඅඅඅඅ"),
    ("very_long_token","a" * 500),
    ("long_sentence",  "තොරතුරු තාක්ෂණය " * 25),
    ("numbers_plain",  "1 2 3 10 25 100 1000 1250 1000000"),
    ("number_date",    "2026 සැප්තැම්බර් 18 වන දින උදෑසන 9.30 ට."),
    ("percentage",     "වට්ටම 15% ක් සහ බදු 8% කි."),
    ("currency",       "මිල රුපියල් 2500 ක් සහ $30 කි."),
    ("acronym_caps",   "CPU GPU RAM ROM USB HDMI ICT WWW"),
    ("acronym_word",   "NASA සහ UNESCO යන ආයතන."),
    ("mixed_script",   "මම Python programming language එක ඉගෙන ගන්නවා."),
    ("english_only",   "The quick brown fox jumps over the lazy dog."),
    ("url_email",      "www.voicelk.lk හෝ test@example.com වෙත ලියන්න."),
    ("symbols_math",   "a + b = c && x <= y || z != w"),
    ("code_snippet",   "for i in range(10): print(i)"),
    ("zwj_conjunct",   "ශ්‍රී ලංකාවේ ක්‍රියාත්මක ව්‍යාපෘතිය."),
    ("unseen_word",    "සුපර්කැලිෆ්‍රැජිලිස්ටික්එක්ස්පියලිඩෝෂස්."),
    ("tamil_script",   "வணக்கம் සහ ආයුබෝවන්."),
]

# Inputs that *should* produce no speech - silence is the correct answer here, so these are
# not counted as failures (otherwise they pollute the error taxonomy in §20).
EXPECTED_EMPTY = {"empty", "whitespace", "emoji_only"}

def silence_ratio(wav, sr, floor_db=-45.0):
    if len(wav) == 0:
        return 1.0
    frames = librosa.util.frame(np.pad(wav, (0, 1024)), frame_length=1024, hop_length=256)
    db = librosa.amplitude_to_db(np.sqrt((frames ** 2).mean(axis=0)) + 1e-10, ref=np.max)
    return float((db < floor_db).mean())

robust_rows = []
for run_name, model in MODELS.items():
    for name, text in EDGE_CASES:
        row = {"run": run_name, "case": name, "input_chars": len(text), "text_preview": text[:45]}
        try:
            processed = TEXT_PIPELINE.process(text)
            row["normalized_preview"] = processed["normalized_text"][:45]
            row["ipa_chars"] = len(processed["ipa_sequence"])
            out = model.synth_ipa(processed["ipa_sequence"])
            row.update({"frontend_ok": True, "synth_ok": out["ok"], "audio_s": out["audio_s"],
                        "tokens": out["n_tokens"], "dropped_chars": "".join(out["dropped_chars"]),
                        "peak": float(np.abs(out["wav"]).max()) if out["wav"].size else 0.0,
                        "silence_ratio": silence_ratio(out["wav"], out["sr"]) if out["wav"].size else 1.0,
                        "chars_per_s": (len(text) / out["audio_s"]) if out["audio_s"] else float("nan"),
                        "error": out["error"]})
        except Exception as exc:
            row.update({"frontend_ok": False, "synth_ok": False, "audio_s": 0.0,
                        "error": f"{type(exc).__name__}: {exc}"})
        flags = []
        row["empty_output_expected"] = name in EXPECTED_EMPTY
        if not row.get("frontend_ok"):
            flags.append("frontend_crash")
        elif name in EXPECTED_EMPTY:
            pass                        # no speakable content in the input; silence is correct
        elif row["input_chars"] > 0 and row.get("ipa_chars", 0) == 0:
            flags.append("text_produced_no_phonemes")
        elif row.get("audio_s", 0) < 0.05 and row["input_chars"] > 3:
            flags.append("no_audio")
        else:
            cps = row.get("chars_per_s", float("nan"))
            if row["input_chars"] > 10 and not math.isnan(cps) and not (3 <= cps <= 40):
                flags.append(f"implausible_rate({cps:.1f}c/s)")
            if row.get("silence_ratio", 0) > 0.8:
                flags.append("mostly_silence")
            if row.get("peak", 0) >= 0.999:
                flags.append("clipping")
        if row.get("dropped_chars"):
            flags.append("chars_dropped")
        row["flags"] = ",".join(flags)
        robust_rows.append(row)

robust_df = pd.DataFrame(robust_rows)
save_table(robust_df, "31_robustness.csv")
for run_name, grp in robust_df.groupby("run"):
    clean = float((grp["flags"] == "").mean())
    add_metric("system", "robustness_clean_rate", run_name, round(clean * 100, 1), "%", n=len(grp),
               note="edge cases producing no anomaly flag", higher_is_better=True)
    add_metric("system", "robustness_hard_failures", run_name,
               int((grp["flags"].str.contains("crash|no_audio|no_phonemes", regex=True, na=False)).sum()),
               "cases", n=len(grp), higher_is_better=False)

print(robust_df.groupby("run")["flags"].apply(lambda s: f"{(s == '').sum()}/{len(s)} clean").to_string())
print("\nflagged cases:")
print(robust_df[robust_df["flags"] != ""][["run", "case", "audio_s", "chars_per_s", "flags"]]
      .to_string(index=False))

---
## 19. Subjective tests — materials, blinding and scoring

Nothing in this section produces a score: MOS, CMOS, AB/ABX, transcription, MUSHRA and the
Turing-style test all need human listeners. What the notebook *can* do is build the material
properly, which is where these tests usually go wrong:

* every stimulus is written out as a `.wav` under `outputs/40_listening_test/`
* rating sheets carry **anonymous sample ids only** — the mapping from id to system lives in a
  separate `*_key.csv` the listener never sees, so the test is blind
* A/B, ABX and MUSHRA orders are randomised per trial with the seed in `CONFIG["random_seed"]`
* MUSHRA includes the hidden reference and a 3.5 kHz low-pass anchor, so listeners who rate the
  hidden reference below ~90 can be screened out
* `listening_test.html` runs the whole thing in a browser and exports the responses as CSV

§19.9 then scores the filled-in sheets: means with 95% confidence intervals, a paired t-test for
CMOS, a binomial test for AB preference, and WER/CER for the transcription test.

Sample-size guidance: ≥15 listeners × ≥10 sentences per system for MOS; CMOS needs fewer
listeners for the same power because it is paired; ABX needs ≥25 trials per listener for the
binomial test to say anything.

In [ ]:
KIT_DIR = os.path.join(OUT_DIR, "40_listening_test")
WAV_DIR = os.path.join(KIT_DIR, "audio")
os.makedirs(WAV_DIR, exist_ok=True)
rng = random.Random(CONFIG["random_seed"])

def write_wav(wav, sr, name):
    """Writes one stimulus, resampled to the common rate so systems cannot be told apart by bandwidth."""
    path = os.path.join(WAV_DIR, name)
    sf.write(path, to_sr(wav, sr), COMMON_SR)
    return os.path.relpath(path, KIT_DIR).replace("\\", "/")

def anon_id(prefix, index):
    return f"{prefix}{index:03d}"

STIMULI = {}      # (run, item_id) -> {"wav": relpath, "text": ..., "source": ...}

if not CONFIG["build_listening_kits"]:
    skip_metric("subjective", "listening-test materials", "disabled in CONFIG['build_listening_kits']",
                "set it to True and re-run")
else:
    for run_name, model in MODELS.items():
        for item in EVAL_SENTENCES:
            out = synth_prompt(run_name, item)
            if not out["ok"] or out["audio_s"] < 0.2:
                continue
            rel = write_wav(out["wav"], out["sr"], f"{run_name[:24]}__{item['id']}.wav")
            STIMULI[(run_name, item["id"])] = {"wav": rel, "text": item["text"],
                                               "category": item["category"], "source": "synth"}
    for run_name, items in REF_PAIRS.items():
        for item in items:
            out = synth_reference(run_name, item)
            if not out["ok"]:
                continue
            key = os.path.splitext(os.path.basename(item["rel_path"]))[0]
            rel = write_wav(out["wav"], out["sr"], f"{run_name[:24]}__ho_{key}.wav")
            STIMULI[(run_name, f"ho_{key}")] = {"wav": rel, "text": item["sinhala"] or item["text"],
                                                "category": "heldout", "source": "synth"}
            ref_rel = write_wav(load_reference_audio(item), COMMON_SR, f"GROUND_TRUTH__ho_{key}.wav")
            STIMULI[("GROUND_TRUTH", f"ho_{key}")] = {"wav": ref_rel, "text": item["sinhala"] or item["text"],
                                                      "category": "heldout", "source": "recording"}
    print(f"{len(STIMULI)} stimuli written to {WAV_DIR}")

### 19.1 MOS — absolute category rating

Each listener hears one stimulus at a time, **without seeing the text**, and rates it 1–5 on three
ITU-T P.800-style axes: naturalness, audio quality, and listening effort (5 = no effort needed).
Systems are interleaved and anonymised.

Intelligibility deliberately is *not* rated here — a listener who can read the sentence always
thinks they understood it. That question belongs to the transcription test in §19.4, and
pronunciation correctness to the error-taxonomy pass in §20, where the text is shown on purpose.

In [ ]:
mos_sheet, mos_key = [], []
for idx, ((run_name, item_id), info) in enumerate(sorted(STIMULI.items())):
    sample_id = anon_id("M", idx)
    mos_sheet.append({"sample_id": sample_id, "audio": info["wav"],
                      "naturalness_1to5": "", "audio_quality_1to5": "", "listening_effort_1to5": "",
                      "notes": ""})
    mos_key.append({"sample_id": sample_id, "system": run_name, "item": item_id,
                    "category": info["category"], "source": info["source"], "text": info["text"]})
rng.shuffle(mos_sheet)

if mos_sheet:
    save_table(pd.DataFrame(mos_sheet), "40_listening_test/mos_rating_sheet.csv")
    save_table(pd.DataFrame(mos_key), "40_listening_test/mos_key.csv")
    print(f"MOS: {len(mos_sheet)} stimuli, {len(set(k['system'] for k in mos_key))} systems "
          f"-> give the sheet + audio/ folder to >=15 listeners")
skip_metric("subjective", "MOS score", "listening test not yet run - only the blinded material is generated",
            "collect mos_rating_sheet.csv from >=15 listeners, then run the scoring cell in 19.9")

### 19.2 CMOS — comparative MOS

The same sentence from two systems, played back to back, rated on a −3…+3 scale (“how much better
is B than A”). Paired, so it detects smaller differences than MOS with the same number of
listeners. Presentation order is randomised and recorded in the key.

In [ ]:
def system_pairs():
    systems = [s for s in MODELS]
    return [(a, b) for i, a in enumerate(systems) for b in systems[i + 1:]]

cmos_sheet, cmos_key = [], []
idx = 0
for sys_a, sys_b in system_pairs():
    for item in EVAL_SENTENCES:
        left, right = (sys_a, sys_b) if rng.random() < 0.5 else (sys_b, sys_a)
        if (left, item["id"]) not in STIMULI or (right, item["id"]) not in STIMULI:
            continue
        pair_id = anon_id("C", idx); idx += 1
        cmos_sheet.append({"pair_id": pair_id,
                           "audio_A": STIMULI[(left, item["id"])]["wav"],
                           "audio_B": STIMULI[(right, item["id"])]["wav"],
                           "score_B_minus_A_-3to+3": "", "notes": ""})
        cmos_key.append({"pair_id": pair_id, "system_A": left, "system_B": right,
                         "item": item["id"], "category": item["category"]})
rng.shuffle(cmos_sheet)

if cmos_sheet:
    save_table(pd.DataFrame(cmos_sheet), "40_listening_test/cmos_sheet.csv")
    save_table(pd.DataFrame(cmos_key), "40_listening_test/cmos_key.csv")
    print(f"CMOS: {len(cmos_sheet)} pairs over {len(system_pairs())} system pair(s). "
          "Scale: -3 = A much better ... 0 = same ... +3 = B much better")
    skip_metric("subjective", "CMOS score", "listening test not yet run - only the blinded material is generated",
                "collect cmos_sheet.csv from listeners, then run the scoring cell in 19.9")
else:
    skip_metric("subjective", "CMOS", "fewer than two models loaded, so there is nothing to compare",
                "load at least two runs")

### 19.3 AB preference and ABX discrimination

**AB** asks which of two systems sounds better — a preference. **ABX** asks whether the listener
can tell them apart at all: X is a copy of either A or B and the listener must say which. If ABX
accuracy is not above chance, a preference result from the same pair means nothing.

In [ ]:
ab_sheet, ab_key, abx_sheet, abx_key = [], [], [], []
idx = 0
for sys_a, sys_b in system_pairs():
    for item in EVAL_SENTENCES:
        if (sys_a, item["id"]) not in STIMULI or (sys_b, item["id"]) not in STIMULI:
            continue
        left, right = (sys_a, sys_b) if rng.random() < 0.5 else (sys_b, sys_a)
        trial_id = anon_id("P", idx); idx += 1
        ab_sheet.append({"trial_id": trial_id,
                         "audio_A": STIMULI[(left, item["id"])]["wav"],
                         "audio_B": STIMULI[(right, item["id"])]["wav"],
                         "preference_A_B_or_none": "", "notes": ""})
        ab_key.append({"trial_id": trial_id, "system_A": left, "system_B": right, "item": item["id"]})

        x_is = "A" if rng.random() < 0.5 else "B"
        abx_sheet.append({"trial_id": trial_id,
                          "audio_A": STIMULI[(left, item["id"])]["wav"],
                          "audio_B": STIMULI[(right, item["id"])]["wav"],
                          "audio_X": STIMULI[((left if x_is == "A" else right), item["id"])]["wav"],
                          "X_matches_A_or_B": ""})
        abx_key.append({"trial_id": trial_id, "system_A": left, "system_B": right,
                        "correct_answer": x_is, "item": item["id"]})

if ab_sheet:
    rng.shuffle(ab_sheet)
    save_table(pd.DataFrame(ab_sheet), "40_listening_test/ab_preference_sheet.csv")
    save_table(pd.DataFrame(ab_key), "40_listening_test/ab_preference_key.csv")
    save_table(pd.DataFrame(abx_sheet), "40_listening_test/abx_sheet.csv")
    save_table(pd.DataFrame(abx_key), "40_listening_test/abx_key.csv")
    print(f"AB: {len(ab_sheet)} trials | ABX: {len(abx_sheet)} trials (>=25 per listener for a usable binomial test)")
    skip_metric("subjective", "AB preference / ABX", "listening test not yet run - only the blinded material is generated",
                "collect ab_preference_sheet.csv and abx_sheet.csv from listeners, then run 19.9")

### 19.4 Intelligibility — transcription test

Listeners type what they hear, with no text on screen. Scoring is WER/CER of the typed transcript
against the sentence that was synthesised, using the same scorer as §10. Real recordings are mixed
in as controls so listener typing noise can be separated from synthesis errors.

In [ ]:
trans_sheet, trans_key = [], []
idx = 0
for (run_name, item_id), info in sorted(STIMULI.items()):
    if not str(info["text"]).strip():
        continue
    sample_id = anon_id("T", idx); idx += 1
    trans_sheet.append({"sample_id": sample_id, "audio": info["wav"],
                        "type_what_you_hear": "", "unintelligible_words": "", "notes": ""})
    trans_key.append({"sample_id": sample_id, "system": run_name, "item": item_id,
                      "source": info["source"], "reference_text": info["text"]})
rng.shuffle(trans_sheet)

if trans_sheet:
    save_table(pd.DataFrame(trans_sheet), "40_listening_test/transcription_sheet.csv")
    save_table(pd.DataFrame(trans_key), "40_listening_test/transcription_key.csv")
    n_ctrl = sum(1 for k in trans_key if k["source"] == "recording")
    print(f"transcription: {len(trans_sheet)} stimuli ({n_ctrl} real-speech controls)")
    skip_metric("subjective", "Intelligibility (human transcription)",
                "listening test not yet run - only the blinded material is generated",
                "collect transcription_sheet.csv from listeners, then run the scorer in 19.9")

### 19.5 MUSHRA

Per trial the listener rates every version of the same sentence 0–100 with the real recording
shown as the labelled reference. The set includes a **hidden reference** (the same recording,
unlabelled) and a **3.5 kHz low-pass anchor**; a listener who does not rate the hidden reference
≥90 or who rates the anchor above a real system is screened out. Needs ground-truth recordings, so
this kit only exists for runs whose dataset audio is on this machine.

In [ ]:
def lowpass_anchor(wav, sr=COMMON_SR, cutoff=3500.0):
    b, a = butter(6, cutoff / (sr / 2), btype="low")
    return filtfilt(b, a, wav).astype(np.float32)

mushra_trials, mushra_key, mushra_sheet = [], [], []
if not REF_PAIRS:
    skip_metric("subjective", "MUSHRA", "needs the real recording of each sentence as the reference, "
                "and no ground-truth audio is available locally",
                "copy the dataset wavs/ folder to this machine and re-run")
else:
    idx = 0
    for run_name, items in REF_PAIRS.items():
        for item in items:
            key_name = os.path.splitext(os.path.basename(item["rel_path"]))[0]
            gt_key = ("GROUND_TRUTH", f"ho_{key_name}")
            if gt_key not in STIMULI:
                continue
            trial_id = anon_id("U", idx); idx += 1
            ref_wav = load_reference_audio(item)
            anchor_rel = write_wav(lowpass_anchor(ref_wav), COMMON_SR, f"ANCHOR__ho_{key_name}.wav")
            conditions = [("hidden_reference", STIMULI[gt_key]["wav"]), ("anchor_3.5kHz", anchor_rel)]
            for other_run in MODELS:
                stim = STIMULI.get((other_run, f"ho_{key_name}"))
                if stim:
                    conditions.append((other_run, stim["wav"]))
            rng.shuffle(conditions)
            labels = [chr(ord("A") + i) for i in range(len(conditions))]
            trial = {"trial_id": trial_id, "reference_audio": STIMULI[gt_key]["wav"]}
            for label, (system, wav_rel) in zip(labels, conditions):
                trial[f"audio_{label}"] = wav_rel
                trial[f"rating_{label}_0to100"] = ""
                mushra_key.append({"trial_id": trial_id, "label": label, "system": system,
                                   "item": key_name})
            mushra_sheet.append(trial)
            mushra_trials.append(trial_id)

    if mushra_sheet:
        save_table(pd.DataFrame(mushra_sheet), "40_listening_test/mushra_sheet.csv")
        save_table(pd.DataFrame(mushra_key), "40_listening_test/mushra_key.csv")
        print(f"MUSHRA: {len(mushra_sheet)} trials x {len(set(k['system'] for k in mushra_key))} conditions "
              "(hidden reference + 3.5 kHz anchor included)")
        skip_metric("subjective", "MUSHRA score", "listening test not yet run - only the blinded material is generated",
                    "collect mushra_sheet.csv from trained listeners, then run the scorer in 19.9")

### 19.6 Turing-style test — real or synthetic?

Real recordings and synthesised versions of the same sentences are shuffled together; the listener
marks each one real or synthetic. 50 % accuracy means indistinguishable; the useful statistic is
how far above chance the listeners score.

In [ ]:
turing_sheet, turing_key = [], []
idx = 0
for (run_name, item_id), info in sorted(STIMULI.items()):
    if info["category"] != "heldout":
        continue                            # only sentences that exist as real recordings
    sample_id = anon_id("R", idx); idx += 1
    turing_sheet.append({"sample_id": sample_id, "audio": info["wav"],
                         "real_or_synthetic": "", "confidence_1to5": ""})
    turing_key.append({"sample_id": sample_id, "truth": "real" if info["source"] == "recording" else "synthetic",
                       "system": run_name, "item": item_id})
rng.shuffle(turing_sheet)

if turing_sheet:
    save_table(pd.DataFrame(turing_sheet), "40_listening_test/turing_sheet.csv")
    save_table(pd.DataFrame(turing_key), "40_listening_test/turing_key.csv")
    n_real = sum(1 for k in turing_key if k["truth"] == "real")
    print(f"Turing test: {len(turing_sheet)} stimuli ({n_real} real, {len(turing_sheet) - n_real} synthetic)")
    skip_metric("subjective", "Turing-style real-vs-synthetic", "listening test not yet run",
                "collect turing_sheet.csv from listeners, then run the scorer in 19.9")
else:
    skip_metric("subjective", "Turing-style real-vs-synthetic",
                "needs real recordings paired with synthesis of the same sentences",
                "copy the dataset wavs/ folder to this machine")

### 19.7 Downstream task — ICT comprehension

VoiceLK's purpose is delivering O/L ICT content by audio, so the end-to-end question is whether a
student can answer questions about a passage they only *heard*. The passages below are synthesised
by every model; the quiz sheet is the instrument. Treat the passages as a starting template and
extend them from the actual syllabus material.

Scoring needs students, so no number is produced here — only the material and the scoring formula
(accuracy per system, plus replay count as a secondary signal).

In [ ]:
ICT_PASSAGES = [
    {"id": "q1", "passage": "පරිගණකයක ප්‍රධාන කොටස් තුනකි. ඒවා නම් ආදාන ඒකක, ක්‍රියාවලි ඒකකය සහ ප්‍රතිදාන ඒකක වේ.",
     "question": "පරිගණකයක ප්‍රධාන කොටස් කීයද?", "options": "එකයි|දෙකයි|තුනයි|හතරයි", "answer": "තුනයි"},
    {"id": "q2", "passage": "RAM යනු තාවකාලික මතක ඒකකයකි. විදුලිය විසන්ධි වූ විට එහි ඇති දත්ත මැකී යයි.",
     "question": "විදුලිය විසන්ධි වූ විට RAM එකේ දත්ත වලට කුමක් සිදුවේද?",
     "options": "ස්ථිරව තැන්පත් වේ|මැකී යයි|දෘඪ තැටියට යයි|වෙනසක් නැත", "answer": "මැකී යයි"},
    {"id": "q3", "passage": "ද්විමය සංඛ්‍යා පද්ධතියේ භාවිතා වන ඉලක්කම් දෙක වන්නේ බින්දුව සහ එක ය.",
     "question": "ද්විමය පද්ධතියේ ඉලක්කම් මොනවාද?",
     "options": "එක සහ දෙක|බින්දුව සහ එක|බින්දුවේ සිට නවය|එකේ සිට දහය", "answer": "බින්දුව සහ එක"},
    {"id": "q4", "passage": "මෘදුකාංග ප්‍රධාන වර්ග දෙකකි. ඒවා පද්ධති මෘදුකාංග සහ යෙදුම් මෘදුකාංග වේ.",
     "question": "මෘදුකාංග ප්‍රධාන වර්ග කීයද?", "options": "දෙකයි|තුනයි|හතරයි|පහයි", "answer": "දෙකයි"},
    {"id": "q5", "passage": "දෘඪ තැටිය ස්ථිර ගබඩා ඒකකයකි. එහි ධාරිතාව සාමාන්‍යයෙන් ගිගාබයිට් වලින් මනිනු ලැබේ.",
     "question": "දෘඪ තැටියේ ධාරිතාව මනිනු ලබන්නේ කුමකින්ද?",
     "options": "හර්ට්ස්|ගිගාබයිට්|වෝල්ට්|පික්සල්", "answer": "ගිගාබයිට්"},
]

quiz_rows, quiz_key = [], []
idx = 0
for run_name, model in MODELS.items():
    for passage in ICT_PASSAGES:
        out = model.synth(passage["passage"])
        if not out["ok"]:
            continue
        rel = write_wav(out["wav"], out["sr"], f"{run_name[:24]}__ict_{passage['id']}.wav")
        sample_id = anon_id("Q", idx); idx += 1
        quiz_rows.append({"sample_id": sample_id, "audio": rel, "question": passage["question"],
                          "options": passage["options"], "student_answer": "", "replays_used": "",
                          "confidence_1to5": ""})
        quiz_key.append({"sample_id": sample_id, "system": run_name, "passage_id": passage["id"],
                         "correct_answer": passage["answer"], "passage": passage["passage"]})
rng.shuffle(quiz_rows)

if quiz_rows:
    save_table(pd.DataFrame(quiz_rows), "40_listening_test/ict_comprehension_sheet.csv")
    save_table(pd.DataFrame(quiz_key), "40_listening_test/ict_comprehension_key.csv")
    print(f"ICT comprehension: {len(quiz_rows)} audio-question items over {len(ICT_PASSAGES)} passages")
skip_metric("subjective", "Downstream ICT comprehension accuracy",
            "needs students to answer the quiz; only the audio + quiz instrument is generated here",
            "run the quiz with >=20 students per system, score accuracy and mean replays per item")

### 19.8 Browser-based listening test

`outputs/40_listening_test/listening_test.html` runs the MOS, AB, ABX, CMOS, MUSHRA,
transcription and Turing tests locally (open it from the same folder so the relative `audio/`
paths resolve) and exports every response as one CSV that §19.9 can score directly.

In [ ]:
def _trial_payload():
    return {
        "mos": [dict(id=r["sample_id"], audio=r["audio"]) for r in mos_sheet],
        "cmos": [dict(id=r["pair_id"], a=r["audio_A"], b=r["audio_B"]) for r in cmos_sheet],
        "ab": [dict(id=r["trial_id"], a=r["audio_A"], b=r["audio_B"]) for r in ab_sheet],
        "abx": [dict(id=r["trial_id"], a=r["audio_A"], b=r["audio_B"], x=r["audio_X"]) for r in abx_sheet],
        "transcription": [dict(id=r["sample_id"], audio=r["audio"]) for r in trans_sheet],
        "turing": [dict(id=r["sample_id"], audio=r["audio"]) for r in turing_sheet],
        "mushra": [dict(id=t["trial_id"], reference=t["reference_audio"],
                        conditions=[{"label": k.split("_")[1], "audio": v}
                                    for k, v in t.items() if k.startswith("audio_")])
                   for t in mushra_sheet],
    }

HTML_TEMPLATE = """<!DOCTYPE html>
<html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>VoiceLK Listening Test</title>
<style>
 :root { --bg:#ffffff; --fg:#16202a; --muted:#5c6b7a; --line:#dde5ec; --accent:#2a6f97; --card:#f6f9fb; }
 @media (prefers-color-scheme: dark) { :root { --bg:#131a21; --fg:#e8eef4; --muted:#9fb0c0; --line:#2a3743; --accent:#7fb7dc; --card:#1a232c; } }
 * { box-sizing:border-box; }
 body { margin:0; padding:0 16px 64px; background:var(--bg); color:var(--fg);
        font:15px/1.55 -apple-system,Segoe UI,Roboto,"Noto Sans Sinhala",sans-serif; }
 header { max-width:900px; margin:0 auto; padding:24px 0 8px; }
 h1 { font-size:22px; margin:0 0 4px; } p.sub { color:var(--muted); margin:0 0 16px; }
 main { max-width:900px; margin:0 auto; }
 nav { display:flex; flex-wrap:wrap; gap:6px; margin-bottom:18px; }
 nav button { border:1px solid var(--line); background:var(--card); color:var(--fg);
              padding:7px 13px; border-radius:999px; cursor:pointer; font-size:13px; }
 nav button.on { background:var(--accent); color:#fff; border-color:var(--accent); }
 .trial { border:1px solid var(--line); background:var(--card); border-radius:12px;
          padding:14px 16px; margin-bottom:12px; }
 .tid { font:12px ui-monospace,monospace; color:var(--muted); }
 audio { width:100%; margin:8px 0; }
 .row { display:flex; flex-wrap:wrap; gap:10px; align-items:center; margin-top:6px; }
 label.sc { font-size:13px; color:var(--muted); }
 input[type=number], input[type=text] { padding:6px 8px; border:1px solid var(--line);
          border-radius:7px; background:var(--bg); color:var(--fg); }
 input[type=text] { width:100%; }
 input[type=range] { width:220px; }
 .bar { position:sticky; bottom:0; background:var(--bg); border-top:1px solid var(--line);
        padding:12px 0; display:flex; gap:10px; align-items:center; }
 .bar button { background:var(--accent); color:#fff; border:0; padding:9px 18px;
               border-radius:8px; cursor:pointer; font-size:14px; }
 .hint { color:var(--muted); font-size:13px; }
</style></head><body>
<header>
  <h1>VoiceLK listening test</h1>
  <p class="sub">Systems are anonymised. Answer what you hear, not what you expect. Enter your
  listener id, work through the tabs, then export the CSV and send it back.</p>
  <div class="row"><label class="sc">Listener id</label><input id="listener" type="text"
       style="max-width:220px" placeholder="e.g. L01"></div>
</header>
<main>
  <nav id="tabs"></nav>
  <div id="panel"></div>
  <div class="bar">
    <button onclick="exportCsv()">Export responses as CSV</button>
    <span class="hint" id="count"></span>
  </div>
</main>
<script>
const TRIALS = __TRIALS_JSON__;
const R = {};
const TABS = [
  ["mos","MOS (1-5)"],["cmos","CMOS (-3..+3)"],["ab","AB preference"],["abx","ABX"],
  ["mushra","MUSHRA (0-100)"],["transcription","Transcription"],["turing","Real or synthetic"]
];
let current = TABS[0][0];
function set(test,id,field,value){ R[test+"|"+id+"|"+field] = value; document.getElementById("count").textContent = Object.keys(R).length + " responses recorded"; }
function player(src){ return '<audio controls preload="none" src="'+src+'"></audio>'; }
function render(){
  document.getElementById("tabs").innerHTML = TABS.map(([k,label]) =>
    '<button class="'+(k===current?"on":"")+'" onclick="current=\\''+k+'\\';render()">'+label+
    ' ('+(TRIALS[k]||[]).length+')</button>').join("");
  const items = TRIALS[current] || [];
  const panel = document.getElementById("panel");
  if (!items.length) { panel.innerHTML = '<p class="hint">No trials of this type were generated.</p>'; return; }
  panel.innerHTML = items.map(t => {
    let body = '<div class="trial"><div class="tid">'+t.id+'</div>';
    if (current === "mos") {
      body += player(t.audio) +
        '<div class="row">' +
        ['naturalness','intelligibility','pronunciation'].map(f =>
          '<label class="sc">'+f+'</label><input type="number" min="1" max="5" step="1" ' +
          'onchange="set(\\'mos\\',\\''+t.id+'\\',\\''+f+'\\',this.value)">').join('') + '</div>';
    } else if (current === "cmos") {
      body += '<div class="hint">A</div>'+player(t.a)+'<div class="hint">B</div>'+player(t.b) +
        '<div class="row"><label class="sc">B minus A</label>' +
        '<input type="number" min="-3" max="3" step="1" onchange="set(\\'cmos\\',\\''+t.id+'\\',\\'score\\',this.value)"></div>';
    } else if (current === "ab") {
      body += '<div class="hint">A</div>'+player(t.a)+'<div class="hint">B</div>'+player(t.b) +
        '<div class="row">' + ['A','B','none'].map(o =>
          '<label class="sc"><input type="radio" name="ab'+t.id+'" value="'+o+'" ' +
          'onchange="set(\\'ab\\',\\''+t.id+'\\',\\'preference\\',\\''+o+'\\')"> '+o+'</label>').join('') + '</div>';
    } else if (current === "abx") {
      body += '<div class="hint">A</div>'+player(t.a)+'<div class="hint">B</div>'+player(t.b) +
        '<div class="hint">X</div>'+player(t.x) +
        '<div class="row">' + ['A','B'].map(o =>
          '<label class="sc"><input type="radio" name="abx'+t.id+'" value="'+o+'" ' +
          'onchange="set(\\'abx\\',\\''+t.id+'\\',\\'x_matches\\',\\''+o+'\\')"> X = '+o+'</label>').join('') + '</div>';
    } else if (current === "mushra") {
      body += '<div class="hint">reference</div>'+player(t.reference) +
        t.conditions.map(c => '<div class="row"><label class="sc">'+c.label+'</label>' + player(c.audio) +
          '<input type="range" min="0" max="100" value="50" ' +
          'oninput="set(\\'mushra\\',\\''+t.id+'\\',\\''+c.label+'\\',this.value)"></div>').join('');
    } else if (current === "transcription") {
      body += player(t.audio) + '<input type="text" placeholder="type exactly what you hear" ' +
        'onchange="set(\\'transcription\\',\\''+t.id+'\\',\\'transcript\\',this.value)">';
    } else if (current === "turing") {
      body += player(t.audio) + '<div class="row">' + ['real','synthetic'].map(o =>
        '<label class="sc"><input type="radio" name="tr'+t.id+'" value="'+o+'" ' +
        'onchange="set(\\'turing\\',\\''+t.id+'\\',\\'answer\\',\\''+o+'\\')"> '+o+'</label>').join('') + '</div>';
    }
    return body + '</div>';
  }).join("");
}
function exportCsv(){
  const listener = (document.getElementById("listener").value || "anonymous").trim();
  const lines = ["listener,test,trial_id,field,value"];
  for (const k in R) { const [test,id,field] = k.split("|");
    lines.push([listener,test,id,field,String(R[k]).replace(/,/g," ")].join(",")); }
  const blob = new Blob([lines.join("\\n")], {type:"text/csv"});
  const a = document.createElement("a");
  a.href = URL.createObjectURL(blob); a.download = "listening_responses_"+listener+".csv"; a.click();
}
render();
</script></body></html>
"""

if STIMULI:
    html = HTML_TEMPLATE.replace("__TRIALS_JSON__", json.dumps(_trial_payload(), ensure_ascii=False))
    html_path = os.path.join(KIT_DIR, "listening_test.html")
    with open(html_path, "w", encoding="utf-8") as fh:
        fh.write(html)
    print("listening test page ->", html_path)
    print("open it from inside 40_listening_test/ so the relative audio paths resolve")

### 19.9 Scoring the returned sheets

Run these once listeners have filled in the sheets (or returned the CSV the HTML page exports).
Each function takes the filled file and its key, and prints the statistic that belongs with that
test: a mean with a 95 % confidence interval for MOS/MUSHRA, a paired t-test for CMOS, a binomial
test for AB and ABX, and WER/CER for transcription.

In [ ]:
from scipy import stats

def mean_ci(values, confidence=0.95):
    values = np.asarray([v for v in values if not (isinstance(v, float) and math.isnan(v))], dtype=float)
    if len(values) < 2:
        return (float(values.mean()) if len(values) else float("nan")), (float("nan"), float("nan"))
    margin = stats.sem(values) * stats.t.ppf((1 + confidence) / 2.0, len(values) - 1)
    return float(values.mean()), (float(values.mean() - margin), float(values.mean() + margin))

def _load(name):
    path = os.path.join(KIT_DIR, name)
    return pd.read_csv(path) if os.path.isfile(path) else None

def score_mos(sheet_name="mos_rating_sheet_filled.csv"):
    sheet, key = _load(sheet_name), _load("mos_key.csv")
    if sheet is None or key is None:
        print("MOS: fill in", sheet_name, "first"); return None
    df = sheet.merge(key, on="sample_id")
    rows = []
    for system, grp in df.groupby("system"):
        for axis in ("naturalness_1to5", "audio_quality_1to5", "listening_effort_1to5"):
            mean, (lo, hi) = mean_ci(pd.to_numeric(grp[axis], errors="coerce").dropna())
            rows.append({"system": system, "axis": axis, "mean": round(mean, 3),
                         "ci95_low": round(lo, 3), "ci95_high": round(hi, 3), "n": int(grp[axis].notna().sum())})
    out = pd.DataFrame(rows)
    save_table(out, "41_mos_scores.csv"); print(out.to_string(index=False)); return out

def score_cmos(sheet_name="cmos_sheet_filled.csv"):
    sheet, key = _load(sheet_name), _load("cmos_key.csv")
    if sheet is None or key is None:
        print("CMOS: fill in", sheet_name, "first"); return None
    df = sheet.merge(key, on="pair_id")
    df["score"] = pd.to_numeric(df["score_B_minus_A_-3to+3"], errors="coerce")
    rows = []
    for (sys_a, sys_b), grp in df.groupby(["system_A", "system_B"]):
        scores = grp["score"].dropna()
        mean, (lo, hi) = mean_ci(scores)
        t_stat, p = stats.ttest_1samp(scores, 0.0) if len(scores) > 1 else (float("nan"), float("nan"))
        rows.append({"system_A": sys_a, "system_B": sys_b, "cmos_B_minus_A": round(mean, 3),
                     "ci95_low": round(lo, 3), "ci95_high": round(hi, 3),
                     "t": round(float(t_stat), 3), "p_value": round(float(p), 4), "n": len(scores)})
    out = pd.DataFrame(rows)
    save_table(out, "41_cmos_scores.csv"); print(out.to_string(index=False)); return out

def score_ab(sheet_name="ab_preference_sheet_filled.csv"):
    sheet, key = _load(sheet_name), _load("ab_preference_key.csv")
    if sheet is None or key is None:
        print("AB: fill in", sheet_name, "first"); return None
    df = sheet.merge(key, on="trial_id")
    df["winner"] = [row.system_A if row.preference_A_B_or_none == "A"
                    else row.system_B if row.preference_A_B_or_none == "B" else "none"
                    for row in df.itertuples()]
    rows = []
    for (sys_a, sys_b), grp in df.groupby(["system_A", "system_B"]):
        wins_a = int((grp["winner"] == sys_a).sum()); wins_b = int((grp["winner"] == sys_b).sum())
        decided = wins_a + wins_b
        p = float(stats.binomtest(wins_a, decided).pvalue) if decided else float("nan")
        rows.append({"system_A": sys_a, "system_B": sys_b, "wins_A": wins_a, "wins_B": wins_b,
                     "no_preference": int((grp["winner"] == "none").sum()),
                     "preference_A_pct": round(100 * wins_a / decided, 1) if decided else float("nan"),
                     "p_value_vs_chance": round(p, 4), "n_decided": decided})
    out = pd.DataFrame(rows)
    save_table(out, "41_ab_preference_scores.csv"); print(out.to_string(index=False)); return out

def score_abx(sheet_name="abx_sheet_filled.csv"):
    sheet, key = _load(sheet_name), _load("abx_key.csv")
    if sheet is None or key is None:
        print("ABX: fill in", sheet_name, "first"); return None
    df = sheet.merge(key, on="trial_id")
    df["correct"] = df["X_matches_A_or_B"].astype(str).str.strip().str.upper() == df["correct_answer"]
    n, hits = len(df), int(df["correct"].sum())
    p = float(stats.binomtest(hits, n, 0.5, alternative="greater").pvalue) if n else float("nan")
    print(f"ABX: {hits}/{n} correct = {100*hits/max(n,1):.1f}%  (p={p:.4f} vs chance; "
          f"{'systems are distinguishable' if p < 0.05 else 'not distinguishable at p<0.05'})")
    out = pd.DataFrame([{"trials": n, "correct": hits, "accuracy_pct": round(100 * hits / max(n, 1), 1),
                         "p_value_vs_chance": round(p, 4)}])
    save_table(out, "41_abx_scores.csv"); return out

def score_transcription(sheet_name="transcription_sheet_filled.csv"):
    sheet, key = _load(sheet_name), _load("transcription_key.csv")
    if sheet is None or key is None:
        print("transcription: fill in", sheet_name, "first"); return None
    df = sheet.merge(key, on="sample_id")
    df["wer"] = [wer(r.reference_text, r.type_what_you_hear) for r in df.itertuples()]
    df["cer"] = [cer(r.reference_text, r.type_what_you_hear) for r in df.itertuples()]
    out = df.groupby(["system", "source"])[["wer", "cer"]].mean().round(3).reset_index()
    save_table(out, "41_transcription_scores.csv"); print(out.to_string(index=False)); return out

def score_mushra(sheet_name="mushra_sheet_filled.csv"):
    sheet, key = _load(sheet_name), _load("mushra_key.csv")
    if sheet is None or key is None:
        print("MUSHRA: fill in", sheet_name, "first"); return None
    long_rows = []
    for row in sheet.to_dict("records"):
        for col, value in row.items():
            if col.startswith("rating_") and str(value).strip() != "":
                long_rows.append({"trial_id": row["trial_id"], "label": col.split("_")[1],
                                  "rating": float(value)})
    df = pd.DataFrame(long_rows).merge(key, on=["trial_id", "label"])
    bad = df[(df["system"] == "hidden_reference") & (df["rating"] < 90)]["trial_id"].unique()
    if len(bad):
        print(f"  post-screening: {len(bad)} trial(s) dropped - hidden reference rated below 90")
        df = df[~df["trial_id"].isin(bad)]
    rows = []
    for system, grp in df.groupby("system"):
        mean, (lo, hi) = mean_ci(grp["rating"])
        rows.append({"system": system, "mean_rating": round(mean, 1), "ci95_low": round(lo, 1),
                     "ci95_high": round(hi, 1), "n": len(grp)})
    out = pd.DataFrame(rows).sort_values("mean_rating", ascending=False)
    save_table(out, "41_mushra_scores.csv"); print(out.to_string(index=False)); return out

def score_turing(sheet_name="turing_sheet_filled.csv"):
    sheet, key = _load(sheet_name), _load("turing_key.csv")
    if sheet is None or key is None:
        print("Turing: fill in", sheet_name, "first"); return None
    df = sheet.merge(key, on="sample_id")
    df["correct"] = df["real_or_synthetic"].astype(str).str.strip().str.lower() == df["truth"]
    n, hits = len(df), int(df["correct"].sum())
    p = float(stats.binomtest(hits, n, 0.5, alternative="greater").pvalue) if n else float("nan")
    print(f"Turing: listeners identified {hits}/{n} = {100*hits/max(n,1):.1f}% correctly (p={p:.4f}); "
          "50% would mean synthesis is indistinguishable from the recordings")
    out = df.groupby(["truth", "system"])["correct"].agg(["mean", "count"]).round(3).reset_index()
    save_table(out, "41_turing_scores.csv"); print(out.to_string(index=False)); return out

def score_ict_quiz(sheet_name="ict_comprehension_sheet_filled.csv"):
    sheet, key = _load(sheet_name), _load("ict_comprehension_key.csv")
    if sheet is None or key is None:
        print("ICT quiz: fill in", sheet_name, "first"); return None
    df = sheet.merge(key, on="sample_id")
    df["correct"] = (df["student_answer"].astype(str).str.strip()
                     == df["correct_answer"].astype(str).str.strip())
    out = df.groupby("system").agg(accuracy=("correct", "mean"),
                                   mean_replays=("replays_used", lambda s: pd.to_numeric(s, errors="coerce").mean()),
                                   n=("correct", "size")).round(3).reset_index()
    save_table(out, "41_ict_comprehension_scores.csv"); print(out.to_string(index=False)); return out

def ingest_html_responses(csv_name):
    """Turns the CSV exported by listening_test.html into the per-test *_filled.csv sheets."""
    path = os.path.join(KIT_DIR, csv_name)
    if not os.path.isfile(path):
        print("not found:", path); return
    resp = pd.read_csv(path)
    mapping = {
        "mos": ("mos_rating_sheet.csv", "sample_id", {"naturalness": "naturalness_1to5",
                "intelligibility": "intelligibility_1to5", "pronunciation": "pronunciation_1to5"}),
        "cmos": ("cmos_sheet.csv", "pair_id", {"score": "score_B_minus_A_-3to+3"}),
        "ab": ("ab_preference_sheet.csv", "trial_id", {"preference": "preference_A_B_or_none"}),
        "abx": ("abx_sheet.csv", "trial_id", {"x_matches": "X_matches_A_or_B"}),
        "transcription": ("transcription_sheet.csv", "sample_id", {"transcript": "type_what_you_hear"}),
        "turing": ("turing_sheet.csv", "sample_id", {"answer": "real_or_synthetic"}),
    }
    for test, (sheet_name, id_col, fields) in mapping.items():
        sub = resp[resp["test"] == test]
        base = _load(sheet_name)
        if sub.empty or base is None:
            continue
        filled = base.copy()
        for field, column in fields.items():
            values = dict(zip(sub[sub["field"] == field]["trial_id"], sub[sub["field"] == field]["value"]))
            filled[column] = filled[id_col].map(values).fillna("")
        save_table(filled, f"40_listening_test/{sheet_name.replace('.csv', '_filled.csv')}")
    mushra_resp = resp[resp["test"] == "mushra"]
    base = _load("mushra_sheet.csv")
    if not mushra_resp.empty and base is not None:
        filled = base.copy()
        for label in sorted(mushra_resp["field"].unique()):
            values = dict(zip(mushra_resp[mushra_resp["field"] == label]["trial_id"],
                              mushra_resp[mushra_resp["field"] == label]["value"]))
            column = f"rating_{label}_0to100"
            if column in filled:
                filled[column] = filled["trial_id"].map(values).fillna("")
        save_table(filled, "40_listening_test/mushra_sheet_filled.csv")
    print("ingested — now call score_mos(), score_cmos(), score_ab(), score_abx(), "
          "score_mushra(), score_transcription(), score_turing()")

print("scoring helpers ready. Typical flow:")
print("  1. listeners fill the sheets (or export CSV from listening_test.html)")
print("  2. ingest_html_responses('listening_responses_L01.csv')   # if using the HTML page")
print("  3. score_mos(); score_cmos(); score_ab(); score_abx(); score_mushra(); "
      "score_transcription(); score_turing(); score_ict_quiz()")

---
## 20. Error taxonomy

Counting failures by *type* is what turns a MOS number into a work plan. The template below is
pre-seeded with everything the notebook could detect automatically (dropped IPA characters,
robustness flags, lexicon misses); the remaining categories need a human listening pass over the
MOS stimuli. `aggregate_error_taxonomy()` turns the filled file into counts per category per
system.

In [ ]:
ERROR_CATEGORIES = [
    ("mispronunciation_sinhala",   "a Sinhala word is pronounced wrongly"),
    ("mispronunciation_english",   "an English/loan word is pronounced wrongly"),
    ("acronym_read_wrong",         "acronym spelled out when it should be read as a word, or vice versa"),
    ("number_read_wrong",          "a number/date/percentage is spoken incorrectly or not at all"),
    ("wrong_language_routing",     "a token was phonemised with the wrong language's rules"),
    ("dropped_phoneme",            "the tokenizer discarded an IPA symbol the front-end produced"),
    ("unnatural_pause",            "pause in the wrong place, or a missing pause at punctuation"),
    ("speaking_rate",              "too fast or too slow for the content"),
    ("prosody_flat",               "monotone / wrong intonation contour"),
    ("audio_artifact",             "buzzing, warbling, metallic or robotic artefacts"),
    ("truncation",                 "the utterance is cut off before the end"),
    ("silence_or_no_output",       "no speech produced for non-empty input"),
    ("speaker_drift",              "the voice changes identity within or between utterances"),
]

taxonomy_rows = []
for name, description in ERROR_CATEGORIES:
    taxonomy_rows.append({"sample_id": "", "system": "", "category": name, "description": description,
                          "severity_1to3": "", "count": "", "evidence": "", "auto_detected": False})

# seed with what was already detected automatically
if "drop_df" in globals() and not drop_df.empty:
    for row in drop_df.itertuples():
        taxonomy_rows.append({"sample_id": "", "system": row.run, "category": "dropped_phoneme",
                              "description": f"tokenizer discards '{row.char}' ({row.unicode} {row.name})",
                              "severity_1to3": 2, "count": int(row.count),
                              "evidence": "20_ipa_dropped_chars.csv", "auto_detected": True})
if "robust_df" in globals():
    for row in robust_df[robust_df["flags"] != ""].itertuples():
        category = ("silence_or_no_output" if "no_audio" in row.flags or "no_phonemes" in row.flags
                    else "speaking_rate" if "implausible_rate" in row.flags
                    else "audio_artifact" if "clipping" in row.flags
                    else "dropped_phoneme" if "chars_dropped" in row.flags else "audio_artifact")
        taxonomy_rows.append({"sample_id": row.case, "system": row.run, "category": category,
                              "description": f"edge case '{row.case}' flagged: {row.flags}",
                              "severity_1to3": 3 if "no_audio" in row.flags else 2, "count": 1,
                              "evidence": "31_robustness.csv", "auto_detected": True})
if "g2p_df" in globals():
    for row in g2p_df[~g2p_df["exact_match"]].sort_values("per", ascending=False).head(20).itertuples():
        taxonomy_rows.append({"sample_id": row.word, "system": "front-end", "category":
                              "mispronunciation_english" if row.lexicon == "en_lexicon" else "mispronunciation_sinhala",
                              "description": f"lexicon says '{row.expected_ipa}', pipeline produced '{row.produced_ipa}'",
                              "severity_1to3": 2, "count": 1,
                              "evidence": "20_g2p_lexicon_fidelity.csv", "auto_detected": True})

taxonomy_df = pd.DataFrame(taxonomy_rows)
save_table(taxonomy_df, "50_error_taxonomy.csv")

def aggregate_error_taxonomy(name="50_error_taxonomy.csv"):
    df = pd.read_csv(os.path.join(OUT_DIR, name))
    df["count"] = pd.to_numeric(df["count"], errors="coerce").fillna(0)
    pivot = df[df["count"] > 0].pivot_table(index="category", columns="system", values="count",
                                            aggfunc="sum", fill_value=0)
    save_table(pivot.reset_index(), "50_error_taxonomy_counts.csv")
    print(pivot.to_string())
    return pivot

auto = int(taxonomy_df["auto_detected"].sum())
print(f"\n{auto} auto-detected issues seeded, {len(ERROR_CATEGORIES)} categories left for the manual pass.")
print("Fill the empty rows while listening to the MOS stimuli, then call aggregate_error_taxonomy().")
aggregate_error_taxonomy()

---
## 21. A/B deployment testing

This one genuinely cannot be run from a notebook: it needs two models serving real users and
telemetry coming back. The spec below defines what to instrument so the test is decidable when
VoiceLK does ship, including the sample size needed to detect a 5 percentage-point change.

In [ ]:
skip_metric("system", "A/B deployment test (engagement, replay, skip rates)",
            "requires both models served to real users with telemetry; there is no deployment or "
            "event pipeline attached to this project yet",
            "ship both models behind a feature flag, log the events in the spec below, and run the "
            "test for at least two weeks or until the required sample size is reached")

def sample_size_for_proportion(baseline=0.30, lift=0.05, alpha=0.05, power=0.80):
    """Per-arm sample size to detect an absolute change in a proportion (two-sided)."""
    z_a, z_b = stats.norm.ppf(1 - alpha / 2), stats.norm.ppf(power)
    p1, p2 = baseline, baseline + lift
    p_bar = (p1 + p2) / 2
    num = (z_a * math.sqrt(2 * p_bar * (1 - p_bar)) + z_b * math.sqrt(p1 * (1 - p1) + p2 * (1 - p2))) ** 2
    return int(math.ceil(num / (lift ** 2)))

ab_spec = [
    {"metric": "listen_through_rate", "definition": "share of playbacks reaching >=90% of the clip",
     "event": "audio_progress", "primary": True, "direction": "higher is better"},
    {"metric": "replay_rate", "definition": "share of clips replayed at least once",
     "event": "audio_replay", "primary": True, "direction": "lower is better (signals unclear audio)"},
    {"metric": "skip_rate", "definition": "share of playbacks abandoned in the first 5 seconds",
     "event": "audio_skip", "primary": True, "direction": "lower is better"},
    {"metric": "session_audio_minutes", "definition": "audio minutes played per session",
     "event": "audio_progress", "primary": False, "direction": "higher is better"},
    {"metric": "thumbs_down_rate", "definition": "explicit negative feedback per 100 clips",
     "event": "feedback_submit", "primary": False, "direction": "lower is better"},
    {"metric": "p95_synthesis_latency", "definition": "server-side p95 time to first audio byte",
     "event": "tts_request", "primary": False, "direction": "lower is better"},
    {"metric": "error_rate", "definition": "failed synthesis requests per 1000",
     "event": "tts_request", "primary": False, "direction": "lower is better"},
]
ab_spec_df = pd.DataFrame(ab_spec)
ab_spec_df["required_n_per_arm_for_5pp"] = [sample_size_for_proportion() if r["primary"] else ""
                                            for r in ab_spec]
save_table(ab_spec_df, "51_ab_deployment_spec.csv")
print(ab_spec_df.to_string(index=False))
print(f"\nSample size: {sample_size_for_proportion()} users per arm to detect a 5 pp change on a "
      "30% baseline (alpha=0.05, power=0.80). Randomise per user, not per request, and keep the "
      "assignment stable across sessions.")

---
## 22. Summary — metric table, skip register and report

In [ ]:
summary_df = pd.DataFrame(RESULTS)
skips_df = pd.DataFrame(SKIPS)
save_table(summary_df, "90_summary_metrics.csv")
save_table(skips_df, "91_skipped_metrics.csv")

pivot = summary_df.pivot_table(index=["group", "metric"], columns="model", values="value", aggfunc="first")
save_table(pivot.reset_index(), "90_summary_pivot.csv")
print(pivot.to_string(), "\n")
print(f"{len(summary_df)} measurements, {len(skips_df)} metrics recorded as skipped")

In [ ]:
def _md_table(df):
    if df.empty:
        return "_none_\n"
    header = "| " + " | ".join(str(c) for c in df.columns) + " |"
    rule = "|" + "|".join("---" for _ in df.columns) + "|"
    rows = ["| " + " | ".join("" if pd.isna(v) else str(v) for v in row) + " |"
            for row in df.itertuples(index=False)]
    return "\n".join([header, rule] + rows) + "\n"

lines = [
    "# VoiceLK TTS — evaluation report",
    "",
    f"Generated {time.strftime('%Y-%m-%d %H:%M')} · device `{DEVICE}` · "
    f"{len(MODELS)} model(s) evaluated · notebook `model_evaluation_full_suite.ipynb`",
    "",
    "## Models",
    "",
    _md_table(registry_df[["run", "run_name", "primary_checkpoint", "sample_rate", "num_chars",
                           "num_speakers", "use_speaker_embedding", "formatter"]]),
    "## Measured metrics",
    "",
]
for group in ["objective", "component", "system", "training", "subjective"]:
    grp = summary_df[summary_df["group"] == group]
    if grp.empty:
        continue
    lines += [f"### {group.capitalize()}", "",
              _md_table(grp[["metric", "model", "value", "unit", "n", "note"]].round(4)), ""]

lines += ["## Metrics not measured, and why", "",
          _md_table(skips_df[["group", "metric", "reason", "what_would_be_needed"]] if not skips_df.empty
                    else skips_df), ""]

lines += [
    "## Human listening tests",
    "",
    "Blinded material for MOS, CMOS, AB, ABX, MUSHRA, transcription, the Turing-style test and the "
    "ICT comprehension quiz is in `outputs/40_listening_test/`. Open `listening_test.html` from "
    "that folder to run them in a browser; the exported CSV feeds `ingest_html_responses()` and "
    "the `score_*()` helpers in §19.9.",
    "",
    "Recommended minimum: 15 listeners × 10 sentences per system for MOS, 25 ABX trials per "
    "listener, and MUSHRA post-screening on the hidden reference.",
    "",
    "## Output files",
    "",
]
for name in sorted(os.listdir(OUT_DIR)):
    path = os.path.join(OUT_DIR, name)
    if os.path.isdir(path):
        lines.append(f"- `{name}/` — {len(os.listdir(path))} files")
    else:
        lines.append(f"- `{name}` — {os.path.getsize(path)/1024:.0f} KB")

report_path = os.path.join(OUT_DIR, "92_evaluation_report.md")
with open(report_path, "w", encoding="utf-8") as fh:
    fh.write("\n".join(lines) + "\n")
print("report ->", report_path)
print("\n".join(lines[:40]))